## Installs e Imports

In [1]:
# Basic settings
from IPython.core.display import HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))
display(HTML("<style>.container { width:100% !important; }</style>"))

import warnings
warnings.filterwarnings("ignore")

In [2]:
#%%capture
!pip install pandas==1.5.3
!pip install numpy==1.24.1
!pip install scipy>=1.9.0
!pip install matplotlib>=3.6.0
!pip install seaborn>=0.12.0
!pip install scikit-learn>=1.2.0
!pip install statsmodels==0.14.1
!pip install plotly==6.0.1
!pip install db_dtypes==1.4.2
!pip install nbformat==5.10.4
!pip install tqdm==4.67.1
!pip install gspread
!pip install oauth2client
!pip install --upgrade google-cloud-bigquery
!pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 107.9 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.0.0
    Uninstalling pandas-2.0.0:
      Successfully uninstalled pandas-2.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 44.5 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.2
    Uninstalling numpy-1.24.2:
      Successfully uninstalled numpy-1.24.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 107.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 MB 80.4 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 2.0.0
    Uninstalling pyarrow-2.0.0:
      Successfully uninstalled pyarrow-2.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour 

In [3]:
%%capture
#Pacotes
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.formula.api as smf
from datetime import datetime, timedelta
# Lake and BQ connections
import pyspark.sql.functions as F
import pyspark.sql.types as st
from pyspark.sql.window import Window
from pyspark.sql import DataFrame as SparkDataFrame
from sness.datalake.sness_spark import SnessSpark
from sness.datalake.metadata import Zone, Environment
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType, StringType, BooleanType
from google.cloud import storage, bigquery
import json
from oauth2client.service_account import ServiceAccountCredentials
import gspread

import seaborn as sns
import scipy.stats as stats
import statsmodels.formula.api as smf
import statsmodels.stats.power as smp
from statsmodels.regression.linear_model import RegressionResultsWrapper
from typing import Sequence
from datetime import date
import random
#from functions_causal import *
#from gera_base_loki_testeAB import gera_base_ab_test
#from functions_conective import *

import contextlib
import io
import itertools
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
import numpy as np
import random
from pyspark import StorageLevel

ss = SnessSpark()
ss.spark.sparkContext.setLogLevel("WARN")

spark = SparkSession.builder \
    .appName("RetailOptimization_CustomVars") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

26/05/28 13:02:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [7]:
import numpy as np, pandas as pd, random, math
from dataclasses import dataclass
from typing import Optional, Tuple, List, Dict

In [5]:
def transform_worksheet_into_spark_dataframe(
    gcs_bucket_name, credentials_folder, doc_id, nome_aba, transpor
):
    """
    Coleta informações do google sheets e converte para um dataframe em Pyspark.
    """

    # Credenciais
    cred = get_credentials_from_gcs(gcs_bucket_name, credentials_folder)
    # Download das infos
    google_worksheet = download_worksheet_from_google_drive(doc_id, cred)
    # Transformação para pyspark
    wsht = google_worksheet.worksheet(nome_aba)
    wsht_pandas = pd.DataFrame(wsht.get_all_values())
    if transpor == True:
        wsht_pandas = wsht_pandas.T
    else:
        pass
    new_header = wsht_pandas.iloc[0]
    wsht_pandas = wsht_pandas[1:]
    wsht_pandas.columns = new_header
    wsht_spark = ss.spark.createDataFrame(wsht_pandas)

    return wsht_spark


def download_worksheet_from_google_drive(doc_id, ssa_credentials_json):
    """
    Efetua a conexão com o google.
    """

    scope = [
        "https://spreadsheets.google.com/feeds",
        "https://www.googleapis.com/auth/drive",
    ]
    credentials = ServiceAccountCredentials.from_json_keyfile_dict(ssa_credentials_json, scope)
    gc = gspread.authorize(credentials)
    gwsht = gc.open_by_key(doc_id)

    return gwsht


def get_credentials_from_gcs(gcs_bucket_name, credentials_folder):
    """
    Obtém as credencias do GCS.
    """

    storage_client = storage.Client()
    bucket = storage_client.get_bucket(gcs_bucket_name)
    blob = storage.Blob(credentials_folder, bucket)
    cred = json.loads(blob.download_as_string().decode("utf-8"))

    return cred


def obtem_dados_sheets_inputs_loki(nome_aba, transpor=False):
    """
    Coleta informação do google sheets e retorna um dataframe em Pyspark.
    """

    BUCKET_NAME = "analytics-labs"
    SSA_CREDENTIALS = "service_account_credentials/analytics.json"
    DOCID = "1kBZxbHr8ZCGY1MWtVNtVKIUx7_LEm5HXeqdm3NQrHy8"

    base = transform_worksheet_into_spark_dataframe(
        BUCKET_NAME, SSA_CREDENTIALS, DOCID, nome_aba, transpor
    )

    return base

def get_base_from_query_bq(query):
    """
    Gera conexão com BQ e obtém tabela a partir de query
    """
    # conexão com o big query
    connection = bigquery.Client()
    query_job = connection.query(query)
    query_job.result()

    # base de dados
    base = (
        ss.spark.read.format("bigquery")
        .option("viewsEnabled", True)
        .option("dataset", query_job.destination.dataset_id)
        .load(query_job.destination.table_id)
    )

    return base


def carrega_df_por_particoes(bucket_name, dataset, n=30):
    """
    Carrega dataset por partições a partir do bucket,
    dataset name e quantidade de partições.
    """

    dataset_list = list(get_dataset(bucket_name, dataset))
    dataset_list = sorted(dataset_list, reverse=True)[0:n]
    dataset_list = [f"gs://{bucket_name}/" + temp for temp in dataset_list]

    df = ss.spark.read.parquet(*dataset_list)

    return df

def get_dataset(bucket_name, dataset):
    """
    Recupera lista com partições de um bucket
    """

    bucket = storage.Client().get_bucket(bucket_name)
    blobs = bucket.list_blobs(prefix=f"{dataset}/", delimiter="/")
    for _ in blobs:
        pass

    return blobs.prefixes

def read_df(
    dataset: str,
    env: str = "prd",
    zone: str = "refined",
    namespace: str = "advanced_analytics",
    **kwargs,
):
    if env == "stg":
        environment = Environment.STAGING
    else:
        environment = Environment.PRODUCTION

    zone_dict = {
        "raw": Zone.RAW,
        "refined": Zone.REFINED,
        "transient": Zone.TRANSIENT,
        "trusted": Zone.TRUSTED,
        "feature": Zone.FEATURE,
    }

    df = ss.read.parquet(
        environment=environment,
        zone=zone_dict[zone],
        namespace=namespace,
        dataset=dataset,
        **kwargs,
    )

    return df

## Parametros Inicias da Base

In [6]:
CATEGORIAS = ["UD", "EP"]
DATA_INICIO = "2026-05-01"
DATA_FIM = "2026-05-02"

## Leitura Bases

## Catalogo

In [ ]:
# Base de catalogo simples, somente necessária para trazer a informação dos produtos como titulo, categoria e entidade.
# Ela é usada como a base primaria de tudo no Join final.

In [8]:
catalogo = (
    read_df(
        env="prd",
        zone="trusted",
        namespace="catalogo",
        dataset="visao_catalogo_produto_atual"
    )
    .where(
        F.col("category_id").isin(CATEGORIAS)
    )
    .select(
        "navigation_id",
        "seller_id",
        "sku",
        F.col("title").alias("descricao"),
        F.col("category_id").alias("categoria"),
        F.col("entity").alias("entidade"),
    )
    .drop_duplicates()
)

In [9]:
SELLERS_BLOQUEADOS = ["magazineluiza", "netshoes", "zattini", "kabum", "epocacosmeticos-integra"]

catalogo = catalogo.filter(~F.col("seller_id").isin(SELLERS_BLOQUEADOS))

## Base Elasticidade + Replica Matching

In [ ]:
# Base com elasticidades criada pelo modelo de ROI Lift

In [10]:
base_elast = read_df(
        env="prd",
        zone="refined",
        namespace="analytics_pricing",
        dataset="pricing_causal_elasticity_coefs",
    )

In [ ]:
## Base de Matching para trazer mais informações de mais produtos similares

In [11]:
base_matching = read_df(
        env="prd",
        zone="refined",
        namespace="catalogo_ds",
        dataset="product_matching",
    )

In [12]:
matching_select = base_matching.select("navigation_id", "seller_id", "matching_hash")

In [ ]:
# Pega o coef mais recente, mas se o desvio padrão do coef for maior que 50% do valor absoluto do coef, troca pela mediana.

In [13]:
#ELASTICIDADE PRÓPRIA — 1 coef por navigation_id
 
w_recente = Window.partitionBy("navigation_id").orderBy(F.desc("date"))
 
# Métricas de qualidade por navigation_id
metricas = (
    base_elast
    .groupBy("navigation_id")
    .agg(
        F.count("*").alias("qtd_observacoes"),
        F.stddev("coef").alias("coef_std"),
        F.expr("percentile_approx(coef, 0.5)").alias("coef_mediana"),
        F.min("date").alias("primeira_data"),
        F.max("date").alias("ultima_data"),
    )
)
 
#Pegar coef mais recente por navigation_id
mais_recente = (
    base_elast
    .withColumn("rn", F.row_number().over(w_recente))
    .filter(F.col("rn") == 1)
    .select("navigation_id", "date", "coef", "intercept")
    .withColumnRenamed("date", "data_elasticidade")
)
 
#Juntar e aplicar regras de qualidade
distinct_elast = (
    mais_recente
    .join(metricas, on="navigation_id", how="left")
    # Se coef variou muito (std > 50% do |coef|), usa mediana
    .withColumn(
        "coef_final",
        F.when(
            (F.col("coef_std").isNotNull())
            & (F.abs(F.col("coef_std")) > F.abs(F.col("coef")) * 0.5),
            F.col("coef_mediana")
        ).otherwise(F.col("coef"))
    )
    # Classificar confiança
    .withColumn(
        "fonte_elasticidade",
        F.when(
            (F.col("qtd_observacoes") >= 2)
            & (
                F.col("coef_std").isNull()
                | (F.abs(F.col("coef_std")) <= F.abs(F.col("coef")) * 0.5)
            ),
            F.lit("confiavel")
        ).when(
            F.col("qtd_observacoes") >= 2,
            F.lit("instavel")
        ).otherwise(
            F.lit("observacao_unica")
        )
    )
    .select(
        "navigation_id",
        F.col("coef_final").alias("coef"),
        "intercept",
        "data_elasticidade",
        "qtd_observacoes",
        "coef_std",
        "fonte_elasticidade",
    )
)

In [ ]:
# Para cada matching_hash, pega o melhor coef disponível (com mais observações e mais recente) e replica para todos os outros navigation_ids do mesmo grupo que não tem coef próprio.

In [14]:
#MATCHING - replicar elasticidade via matching_hash
 
#Para cada matching_hash, pegar o melhor coef disponível
w_hash = (
    Window.partitionBy("matching_hash")
    .orderBy(F.desc("qtd_observacoes"), F.desc("data_elasticidade"))
)
 
hash_com_elast = (
    matching_select
    .join(distinct_elast, on="navigation_id", how="inner")
    .withColumn("rn", F.row_number().over(w_hash))
    .filter(F.col("rn") == 1)
    .select(
        "matching_hash",
        F.col("coef").alias("coef_matching"),
        F.col("intercept").alias("intercept_matching"),
        F.col("fonte_elasticidade").alias("fonte_elast_matching"),
        F.col("data_elasticidade").alias("data_elast_matching"),
        F.col("qtd_observacoes").alias("qtd_obs_matching"),
        F.col("coef_std").alias("coef_std_matching"),
    )
)
 
#Nav_ids que JÁ TÊM elasticidade própria
elast_propria = (
    matching_select
    .join(distinct_elast, on="navigation_id", how="inner")
    .withColumn("origem_elasticidade", F.lit("proprio"))
    .select(
        "navigation_id",
        "seller_id",
        "matching_hash",
        "coef",
        "intercept",
        "data_elasticidade",
        "qtd_observacoes",
        "coef_std",
        "fonte_elasticidade",
        "origem_elasticidade",
    )
)
 
#Nav_ids SEM elasticidade própria vai buscar via matching_hash
sem_elast = (
    matching_select
    .join(distinct_elast, on="navigation_id", how="left_anti")
)
 
com_matching = (
    sem_elast
    .join(hash_com_elast, on="matching_hash", how="inner")
    .select(
        "navigation_id",
        "seller_id",
        "matching_hash",
        F.col("coef_matching").alias("coef"),
        F.col("intercept_matching").alias("intercept"),
        F.col("data_elast_matching").alias("data_elasticidade"),
        F.col("qtd_obs_matching").alias("qtd_observacoes"),
        F.col("coef_std_matching").alias("coef_std"),
        F.col("fonte_elast_matching").alias("fonte_elasticidade"),
        F.lit("matching").alias("origem_elasticidade"),
    )
)

In [15]:
#UNION FINAL — base de elasticidade completa
 
base_elasticidade_final = (
    elast_propria
    .unionByName(com_matching)
)

In [16]:
base_elasticidade_final = (
    base_elasticidade_final
    .filter(F.lower(F.col("seller_id")) != "magazineluiza")
)

In [ ]:
# Coluna fonte elasticidade/origem elasticidade
#########
# proprio pode ter 3 status:
# confiavel - melhor cenário
# instavel - coef proprio mas variou muito durante
# observacao_unica - só tem 1 data
#########
# matching pode ter 3 status:
# confiavel - replicado de um coef bom
# instavel - replicado de um coef que variou
# observacao_unica - replicado de 1 só uma data

In [17]:
base_elasticidade_final = base_elasticidade_final.cache()
base_elasticidade_final.show()

26/05/28 13:29:18 ERROR TaskSchedulerImpl: Lost executor 1 on 10.251.200.194: ]]
The executor with id 1 exited with exit code 143(unexpected).
The API gave the following brief reason: Terminated
The API gave the following message: Pod was terminated in response to imminent node shutdown.

The API gave the following container statuses:


	 container name: spark-kubernetes-executor
	 container image: gcr.io/maga-bigdata/dataops/cloud-magalu/spark-jupyter-pyspark-arcade-gcp-3.2.4:1.2.1
	 container state: terminated
	 container started at: 2026-05-28T13:03:55Z
	 container finished at: 2026-05-28T13:29:16Z
	 exit code: 143
	 termination reason: Error
      
[Stage 36:============(333 + -21) / 312][Stage 41:>                 (0 + 1) / 1]]

+-------------+--------------------+--------------------+--------------------+--------------------+-----------------+---------------+--------------------+------------------+-------------------+
|navigation_id|           seller_id|       matching_hash|                coef|           intercept|data_elasticidade|qtd_observacoes|            coef_std|fonte_elasticidade|origem_elasticidade|
+-------------+--------------------+--------------------+--------------------+--------------------+-----------------+---------------+--------------------+------------------+-------------------+
|    476762500|       livrarialuana|4928e6bad772a3fd1...|-0.01223622163361...| -7.7727464468238905|       2026-05-25|              1|                null|  observacao_unica|            proprio|
|    501295000|apaginadistribuid...|6b6368baad2db275d...|                 0.0|0.002842163947726...|       2026-03-31|             21|0.010782901376499699|          instavel|            proprio|
|    506415600|ponttolavado-op

In [18]:
base_elasticidade_final.count()

13083458

[Stage 36:====================================================(333 + -21) / 312]

In [21]:
#Cobertura
 
print("=" * 60)
print("  COBERTURA DE ELASTICIDADE")
print("=" * 60)
 
base_elasticidade_final.groupBy(
    "origem_elasticidade", "fonte_elasticidade"
).count().orderBy(
    "origem_elasticidade", "fonte_elasticidade"
).show(truncate=False)
 
total = base_elasticidade_final.count()
por_origem = base_elasticidade_final.groupBy("origem_elasticidade").count().collect()
for row in por_origem:
    pct = (row["count"] / total) * 100
    print(f"  {row['origem_elasticidade']:20s}: {row['count']:>8,} ({pct:.1f}%)")
 
print("=" * 60)

  COBERTURA DE ELASTICIDADE


+-------------------+------------------+-------+
|origem_elasticidade|fonte_elasticidade|count  |
+-------------------+------------------+-------+
|matching           |confiavel         |5811380|
|matching           |instavel          |4268883|
|matching           |observacao_unica  |1138375|
|proprio            |confiavel         |894678 |
|proprio            |instavel          |439298 |
|proprio            |observacao_unica  |266825 |
+-------------------+------------------+-------+



[Stage 53:====================================================(220 + -20) / 200]

  proprio             : 1,600,801 (12.5%)
  matching            : 11,218,638 (87.5%)


## Vendas Captadas

In [ ]:
# Base de vendas que utilizamos somente para trazer a informação de quantidade de itens vendidos, por data, numero do pedido cliente, navigation_id e seller_id.

In [19]:
base_vendas_raw = (
    ss.read.parquet(
        environment="prd",
        zone="trusted",
        namespace="mlpbi",
        dataset="mag_t_fact_vendas_captadas",
    )
    .where(
        (F.col("datapedidokey") >= int(DATA_INICIO.replace("-", "")))
        & (F.col("datapedidokey") <= int(DATA_FIM.replace("-", "")))
        & (F.col("produtositekey") == "000000007")
        & (F.col("produtomarketplacekey") != "-1")
        & (F.col("status") != "Teste")
        & (F.col("categoria").isin(CATEGORIAS))
    )
)

[Stage 36:====================================================(333 + -21) / 312]

In [20]:
base_vendas = (
    base_vendas_raw
    .select(
        F.to_date(F.col("datapedidokey").cast("string"), "yyyyMMdd").alias("date"),
        F.col("produtomarketplacekey").alias("navigation_id"),
        F.col("idsellermarketplace").alias("seller_id"),
        F.col("numeropedidocliente"),
        F.col("categoria"),
        F.col("quantidadeitens"),
    )
)

## IP

In [ ]:
# Base de IP a prazo, já filtrada com as categorias

In [21]:
query = f"""
SELECT DataPreco, HoraPreco, NavigationID as navigation_id, Carteira, IP, Concorrente
FROM (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY NavigationID 
            ORDER BY DataPreco DESC, HoraPreco DESC
        ) AS rn
    FROM `maga-bigdata.bi_pricing.tbl_pricing_ip_hora_a_prazo`
)
WHERE rn = 1
"""

ip = get_base_from_query_bq(query)

In [22]:
ip = ip.cache()
ip.show()

[Stage 36:============(333 + -21) / 312][Stage 69:>                 (0 + 1) / 1]

+----------+---------+-------------+------------+------------------+---------------+
| DataPreco|HoraPreco|navigation_id|    Carteira|                IP|    Concorrente|
+----------+---------+-------------+------------+------------------+---------------+
|2025-07-09|       23|    439901100|           -|0.9679032930387662|Google Shopping|
|2026-04-20|       16|    501382200|       Epoca| 1.190608695652174|Google Shopping|
|2026-04-17|       23|    590071300|           -|0.9375029258929825|  Mercado Livre|
|2026-05-06|       23|   aa31egghea|Autosserviço|  0.99999998027784|Madeira Madeira|
|2026-04-15|       23|   aa5eece7dh|Assessorados|0.5662337662337662|  Beleza na Web|
|2025-01-10|       23|   aad88753b1|           -|1.0308237861429352|     Americanas|
|2025-12-25|       23|   aajbc41fk1|Assessorados|0.7548886066733715|  Mercado Livre|
|2026-01-31|       23|   ac3h3273e1|Assessorados|1.5738866396761133|         Amazon|
|2025-01-29|       23|   acg608b5fk|           -| 1.2405436060729

## Estoque

In [ ]:
# Base de como estava ontem o status de estoque dos navigation_id.

In [23]:
query = f"""
SELECT * FROM (
    SELECT
        it.product_navigation_id,
        it.product_organization,
        coalesce((select b.value from unnest(amount.key_value) as b where b.key = 'physicLegacy'),0) as physic_amount,
        (
            coalesce((select b.value from unnest(reserved.key_value) as b where b.key = 'physic'),0) + 
            coalesce((select b.value from unnest(reserved.key_value) as b where b.key = 'physicLegacy'),0)
        ) as physic_reserved,
        (
            coalesce((select b.value from unnest(amount.key_value) as b where b.key = 'logic'),0) + 
            coalesce((select b.value from unnest(amount.key_value) as b where b.key = 'logicLegacyReserved'),0)
        ) as logic_amount,
        coalesce((select b.value from unnest(amount.key_value) as b where b.key = 'logicLegacyReserved'),0) as logic_reserved,
        coalesce((select b.value from unnest(amount.key_value) as b where b.key = 'technical'),0) as technical_amount,
        it.date
    FROM `maga-bigdata.daenerys.inventory_daily_position_events` it
) d
WHERE d.product_organization != 'fulfillment'
AND date = DATE_SUB(CURRENT_DATE('America/Sao_Paulo'), INTERVAL 1 DAY)
"""

estoque = get_base_from_query_bq(query)

[Stage 36:====================================================(333 + -21) / 312]

In [24]:
estoque = estoque.cache()
estoque.show()

[Stage 36:============(333 + -21) / 312][Stage 70:>                 (0 + 1) / 1]

+---------------------+--------------------+-------------+---------------+------------+--------------+----------------+----------+
|product_navigation_id|product_organization|physic_amount|physic_reserved|logic_amount|logic_reserved|technical_amount|      date|
+---------------------+--------------------+-------------+---------------+------------+--------------+----------------+----------+
|           kefkh1da3a|      fastvarejoloja|           87|              1|           0|             0|               0|2026-05-27|
|           ch3c2c20ck|         zzstoreltda|           51|              0|           0|             0|               0|2026-05-27|
|           bdah1gc8g0|xbtcomercioedistr...|           23|              0|           0|             0|               0|2026-05-27|
|           bd18bbe77b|         lojadocthos|           37|              0|           0|             0|               0|2026-05-27|
|           ajebj151kc|   oficialamericanas|         2732|              0|         

In [25]:
estoque = (
    estoque
    .where(F.col("product_organization") != "magazineluiza")
    .groupBy(
        "date",
        F.col("product_organization").alias("seller_id"),
        F.col("product_navigation_id").alias("navigation_id"),
    )
    .agg(
        F.sum(F.col("physic_amount") - F.coalesce(F.col("physic_reserved"), F.lit(0))).alias("estoque")
    )
)

## Preço, Rebate e Campanha

In [ ]:
# Base de rebate que pego as informações de preço por do navigation_id e o rebate total utilizado no pedido com aquele produto.
# Informações de campanhas faz duplicar a quantidade de linhas então tive que fazer um collect_set para conseguir utilizar no otimizador.

In [26]:
query = f"""
SELECT * FROM `maga-bigdata.bi_pricing.vw_promo_rebate_consolidado`
"""

rebate = get_base_from_query_bq(query)

[Stage 36:====================================================(333 + -21) / 312]

In [27]:
base_rebate = (
    rebate
    .withColumnRenamed("Data", "date")
    .withColumnRenamed("pedido_cliente", "numeropedidocliente")
    .withColumnRenamed("id_seller", "seller_id")
    .groupBy("date", "numeropedidocliente", "navigation_id", "seller_id")
    .agg(
        F.sum("rebate").alias("rebate_total"),
        F.first(F.col("preco_por"), ignorenulls=True).alias("preco_por"),
        F.collect_set("id_promocao").alias("id_campanhas"),
        F.collect_set("nome_promocao").alias("nome_campanhas"),
    )
)

In [33]:
# Join vendas + rebate (granular, precisa do pedido)
vendas_com_rebate = (
    base_vendas
    .join(
        base_rebate,
        on=["date", "numeropedidocliente", "navigation_id", "seller_id"],
        how="left"
    )
)

# Depois agrega (sem numeropedidocliente)
vendas_agg = (
    vendas_com_rebate
    .groupBy("navigation_id", "seller_id")
    .agg(
        F.sum("quantidadeitens").alias("quantidadeitens"),
        F.sum("rebate_total").alias("rebate_gasto_total"),
        F.round(
            F.sum("rebate_total") / F.sum("quantidadeitens"), 2
        ).alias("rebate_unitario_medio"),
        F.first(F.col("preco_por"), ignorenulls=True).alias("preco_por"),
    )
)

[Stage 36:====================================================(333 + -21) / 312]

## Base Preço Promotions

In [ ]:
# Base extra de preço de produtos, pois quando fiz os joins a cobertura de navigation_id com preço nulo estava alta, pois a base de rebate só traz os pedidos que tiveram rebate utilizado.
# O otimizador precisa de produtos que tenham potencial, que não foi utilizado rebate e o IP está ruim, então achei viavel fazer um fallback de mais uma base de preço para complementar.

In [28]:
query = f"""
SELECT 
    sku.metadata.id_marketplace AS navigation_id,
    id_seller AS seller_id,
    sku.id AS sku_id,
    sku.price AS preco_por,
    sku.list_price AS preco_de,
    updated_at
FROM `maga-bigdata.promocional3p.promotion_skus`
WHERE sku.price IS NOT NULL
AND sku.price > 0
AND sku.metadata.id_marketplace IS NOT NULL
AND updated_at >= TIMESTAMP('2026-01-01')
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY sku.metadata.id_marketplace, id_seller 
    ORDER BY updated_at DESC
) = 1
"""

preco = get_base_from_query_bq(query)

In [45]:
preco.show()

+-------------+-------------------+--------------------+---------+--------+--------------------+
|navigation_id|          seller_id|              sku_id|preco_por|preco_de|          updated_at|
+-------------+-------------------+--------------------+---------+--------+--------------------+
|   hk1d121d66|       rtmbikesltda|  SAPA-ROAD-53949-45|   505.98|  505.98|2026-04-29 13:16:...|
|   hk8941kafb|     blackwatchltda|    DBZ-SORTIDOS-164|    69.92|   69.92|2026-05-22 14:11:...|
|   ja1bg0e5a1|    imperiodoquadro|MG-DP-120x75-MLD-...|    849.0|   849.0|2026-05-22 14:59:...|
|   jb66b2fkc8|         sacolaocom|              328937|     29.9|    29.9|2026-05-19 17:35:...|
|   jbjj9ge7b0|           lumesexy|ffe3b11eb6e211edb...|     18.9|    18.9|2026-01-23 13:40:...|
|   jc9c1c2g01|          olistplus|    OPM0BC9BZ48BPPBU|   105.37|  105.37|2026-03-26 21:58:...|
|   jc9f45k513|          uppremium|              338109|    172.9|   172.9|2026-04-29 13:16:...|
|   jcb45189k5|casadafamadesig

In [29]:
base_preco_fallback = (
    preco
    .select(
        "navigation_id",
        "seller_id",
        F.col("preco_por").alias("preco_por_fallback"),
    )
)

## Visão Unica Prices

In [45]:
base_visao_prices = (
    ss.read.parquet(
        environment="prd",
        zone="refined",
        namespace="visao_unica",
        dataset="prices_historic",
    )
    .where(
        & (F.col("seller_id") != "magazineluiza")
        & (F.col("active") != "false")
    )

SyntaxError: invalid syntax (490443672.py, line 9)

In [43]:
base_visao_prices.printSchema()

root
 |-- price_timestamp: timestamp (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- navigation_id: string (nullable = true)
 |-- sku: string (nullable = true)
 |-- active: boolean (nullable = true)
 |-- stock: boolean (nullable = true)
 |-- price_de: decimal(8,2) (nullable = true)
 |-- price_por: decimal(8,2) (nullable = true)
 |-- price_pix: decimal(8,2) (nullable = true)
 |-- discount_perc_pix: decimal(8,2) (nullable = true)
 |-- payment_method_campaign: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- price_boleto: decimal(8,2) (nullable = true)
 |-- price_magalupay: decimal(8,2) (nullable = true)
 |-- discount_perc: decimal(8,2) (nullable = true)
 |-- discount_perc_ouro: decimal(8,2) (nullable = true)
 |-- price_por_ouro: decimal(8,2) (nullable = true)
 |-- price_pix_ouro: decimal(8,2) (nullable = true)
 |-- installment_price_once_visa: decimal(8,2) (nullable = true)
 |-- installment_price_on

## Visitas

In [30]:
base_visitas = (
    ss.read.parquet(
        environment="prd",
        zone="refined",
        namespace="web_analytics",
        dataset="ds_ga4",
    )
    .where(
        (F.col("date") >= DATA_INICIO) &
        (F.col("date") <= DATA_FIM)
    )
    .groupBy("item_id")
    .agg(
        F.countDistinct("sessionid").alias("qtd_visitas")
    )
    .withColumnRenamed("item_id", "navigation_id")
)

[Stage 36:====================================================(333 + -21) / 312]

## Joins (antigo)

In [51]:
vendas_agg.show()

[Stage 53:====================================================(220 + -20) / 200]5]]

+-------------+--------------------+---------------+------------------+---------------------+-------------+
|navigation_id|           seller_id|quantidadeitens|rebate_gasto_total|rebate_unitario_medio|    preco_por|
+-------------+--------------------+---------------+------------------+---------------------+-------------+
|   kbe4ed1092|purificadoresdeag...|              1|              null|                 null|         null|
|   jg3jc43g74|           starhouse|              6|              null|                 null|         null|
|   gefg7a50kk| rodrigues-comprebem|              2|              0E-9|                 0.00| 52.900000000|
|   fj8dkdjk2j|      newshoppeletro|              5|     384.565000000|                76.91|         null|
|   hgd702fhbe|donalaurautilidad...|              3|       7.356000000|                 2.45| 37.780000000|
|   bbe334600a|            mimorada|             81|    3155.240000000|                38.95|599.900000000|
|   fkac962fe0|            c

26/05/26 18:55:34 ERROR TaskSchedulerImpl: Lost executor 10 on 10.251.219.2: 00]
The executor with id 10 exited with exit code 143(unexpected).
The API gave the following brief reason: Terminated
The API gave the following message: Pod was terminated in response to imminent node shutdown.

The API gave the following container statuses:


	 container name: spark-kubernetes-executor
	 container image: gcr.io/maga-bigdata/dataops/cloud-magalu/spark-jupyter-pyspark-arcade-gcp-3.2.4:1.2.1
	 container state: terminated
	 container started at: 2026-05-26T18:32:27Z
	 container finished at: 2026-05-26T18:55:34Z
	 exit code: 143
	 termination reason: Error
      
[Stage 53:====================================================(220 + -20) / 200]

In [37]:
df_final = (
    catalogo
    .join(base_elasticidade_final, on=["navigation_id", "seller_id"], how="inner")
    .join(estoque, on=["navigation_id", "seller_id"], how="inner")
    .join(ip, on="navigation_id", how="left")
    .join(vendas_agg, on=["navigation_id", "seller_id"], how="inner")
    .join(base_preco_fallback, on=["navigation_id", "seller_id"], how="left")
    .join(base_visitas, on="navigation_id", how="left")
    .withColumn(
        "preco_por",
        F.coalesce(F.col("preco_por"), F.col("preco_por_fallback"))
    )
    .drop("preco_por_fallback")
    .withColumn("qtd_visitas", F.coalesce(F.col("qtd_visitas"), F.lit(0)))
    .withColumn("IP", F.coalesce(F.col("IP"), F.lit(1.0)))
    .withColumn("concorrente", F.coalesce(F.col("concorrente"), F.lit("N/A")))
    .withColumn("carteira", F.coalesce(F.col("carteira"), F.lit("N/A")))
    .withColumn(
        "preco_concorrente",
        F.when(F.col("IP") > 0, F.round(F.col("preco_por") / F.col("IP"), 2))
        .otherwise(F.col("preco_por"))
    )
    .select(
        "navigation_id",
        "seller_id",
        "matching_hash",
        "sku",
        "descricao",
        "categoria",
        "entidade",
        "carteira",
        "concorrente",
        "coef",
        "intercept",
        "data_elasticidade",
        "qtd_observacoes",
        "coef_std",
        "fonte_elasticidade",
        "origem_elasticidade",
        "preco_por",
        "IP",
        "preco_concorrente",
        "quantidadeitens",
        "estoque",
        "rebate_gasto_total",
        "rebate_unitario_medio",
        "qtd_visitas",
    )
)

In [38]:
# Filtrar produtos prontos pro otimizador
df_final = (
    df_final
    .filter(
        (F.col("preco_por").isNotNull())
        & (F.col("preco_por") > 0)
        & (F.col("estoque") > 0)
        & (F.col("quantidadeitens") > 0)
        & (F.col("carteira").isin("-", "autosserviço", "Autosserviço", "Engajamento", "N/A"))
    )
)

print("Produtos prontos:", df_final.count())

[Stage 291:===================================================>  (95 + 5) / 100]]5]

Produtos prontos: 36327


In [53]:
df_final.show()

[Stage 252:========(2184 + -184) / 2000][Stage 668:=============> (66 + 9) / 75]/ 50]]]0]

+-------------+-------------------+--------------------+--------------------+---------+--------------------+-----------+-------------+-------------------+--------------------+-----------------+---------------+-------------------+------------------+-------------------+---------+------------------+-----------------+---------------+-------+------------------+---------------------+
|navigation_id|          seller_id|       matching_hash|           descricao|categoria|            entidade|   carteira|  concorrente|               coef|           intercept|data_elasticidade|qtd_observacoes|           coef_std|fonte_elasticidade|origem_elasticidade|preco_por|                IP|preco_concorrente|quantidadeitens|estoque|rebate_gasto_total|rebate_unitario_medio|
+-------------+-------------------+--------------------+--------------------+---------+--------------------+-----------+-------------+-------------------+--------------------+-----------------+---------------+-------------------+---------

[Stage 252:================================================(2184 + -184) / 2000]

## Base Dia e Mensal

In [ ]:
#Divisao de bases para separação de configurações de rebate diário e rebate mensal.

In [31]:
CATEGORIAS = ["UD", "EP"]
DATA_INICIO = "2026-05-01"
DATA_FIM = "2026-05-02"
ORCAMENTO_MENSAL = 800_000
ORCAMENTO_DIARIO = ORCAMENTO_MENSAL / 30 
MAX_DESCONTO = 0.10
IP_ALVO = 0.98

In [34]:
vendas_agg_mensal = (
    vendas_com_rebate
    .groupBy("navigation_id", "seller_id")
    .agg(
        F.sum("quantidadeitens").alias("quantidadeitens"),
        F.sum("rebate_total").alias("rebate_gasto_total"),
        F.when(
            F.sum("quantidadeitens") > 0,
            F.round(F.sum("rebate_total") / F.sum("quantidadeitens"), 2)
        ).otherwise(F.lit(0.0)).alias("rebate_unitario_medio"),
        F.first(F.col("preco_por"), ignorenulls=True).alias("preco_por_vendas"),
    )
)

vendas_agg_diario = (
    vendas_com_rebate
    .groupBy("navigation_id", "seller_id")
    .agg(
        F.round(
            F.sum("quantidadeitens") / F.countDistinct("date"), 2
        ).alias("quantidadeitens"),
        F.round(
            F.sum("rebate_total") / F.countDistinct("date"), 2
        ).alias("rebate_gasto_total"),
        F.when(
            F.sum("quantidadeitens") > 0,
            F.round(F.sum("rebate_total") / F.sum("quantidadeitens"), 2)
        ).otherwise(F.lit(0.0)).alias("rebate_unitario_medio"),
        F.first(F.col("preco_por"), ignorenulls=True).alias("preco_por_vendas"),
    )
)

In [35]:
base_fixa = (
    catalogo
    .join(base_elasticidade_final, on=["navigation_id", "seller_id"], how="inner")
    .join(estoque, on=["navigation_id", "seller_id"], how="inner")
    .join(ip, on="navigation_id", how="left")
    .join(base_preco_fallback, on=["navigation_id", "seller_id"], how="left")
    .withColumn("IP", F.coalesce(F.col("IP"), F.lit(1.0)))
    .withColumn("concorrente", F.coalesce(F.col("concorrente"), F.lit("N/A")))
    .withColumn("carteira", F.coalesce(F.col("carteira"), F.lit("N/A")))
)

In [36]:
def montar_df_final(base_fixa, vendas_agg, categoria=None):
    df = (
        base_fixa
        .join(vendas_agg, on=["navigation_id", "seller_id"], how="inner")
        .join(base_visitas, on="navigation_id", how="left")
        .withColumn(
            "preco_por",
            F.coalesce(F.col("preco_por_vendas"), F.col("preco_por_fallback"))
        )
        .drop("preco_por_fallback", "preco_por_vendas")
        .withColumn("qtd_visitas", F.coalesce(F.col("qtd_visitas"), F.lit(0)))
        .filter(
            (F.col("preco_por").isNotNull())
            & (F.col("preco_por") > 0)
            & (F.col("estoque") > 0)
            & (F.col("quantidadeitens") > 0)
            & (F.col("carteira").isin("-", "autosserviço", "Autosserviço", "Engajamento", "N/A"))
        )
    )
    if categoria:
        df = df.filter(F.col("categoria") == categoria)
    return df

In [37]:
df_ud_mensal = montar_df_final(base_fixa, vendas_agg_mensal, "UD")
df_ud_diario = montar_df_final(base_fixa, vendas_agg_diario, "UD")
df_ep_mensal = montar_df_final(base_fixa, vendas_agg_mensal, "EP")
df_ep_diario = montar_df_final(base_fixa, vendas_agg_diario, "EP")

#Check se a quantidade de produtos em cada base.
print("UD mensal:", df_ud_mensal.count())
print("UD diário:", df_ud_diario.count())
print("EP mensal:", df_ep_mensal.count())
print("EP diário:", df_ep_diario.count())

                                                                                ]]]

UD mensal: 1597


                                                                                ]]]0]

UD diário: 1597


26/05/28 13:40:25 ERROR TaskSchedulerImpl: Lost executor 3 on 10.251.199.98:  50]]]0]
The executor with id 3 exited with exit code 143(unexpected).
The API gave the following brief reason: Terminated
The API gave the following message: Pod was terminated in response to imminent node shutdown.

The API gave the following container statuses:


	 container name: spark-kubernetes-executor
	 container image: gcr.io/maga-bigdata/dataops/cloud-magalu/spark-jupyter-pyspark-arcade-gcp-3.2.4:1.2.1
	 container state: terminated
	 container started at: 2026-05-28T13:04:05Z
	 container finished at: 2026-05-28T13:40:24Z
	 exit code: 143
	 termination reason: Error
      
26/05/28 13:40:33 ERROR TaskSchedulerImpl: Lost executor 4 on 10.251.198.226: 0]]
The executor with id 4 exited with exit code 143(unexpected).
The API gave the following brief reason: Terminated
The API gave the following message: Pod was terminated in response to imminent node shutdown.

The API gave the following container status

EP mensal: 725


26/05/28 13:45:20 ERROR TaskSchedulerImpl: Lost executor 2 on 10.251.199.66: 50]]]]0]]
The executor with id 2 exited with exit code 143(unexpected).
The API gave the following brief reason: Terminated
The API gave the following message: Pod was terminated in response to imminent node shutdown.

The API gave the following container statuses:


	 container name: spark-kubernetes-executor
	 container image: gcr.io/maga-bigdata/dataops/cloud-magalu/spark-jupyter-pyspark-arcade-gcp-3.2.4:1.2.1
	 container state: terminated
	 container started at: 2026-05-28T13:03:58Z
	 container finished at: 2026-05-28T13:45:19Z
	 exit code: 143
	 termination reason: Error
      
[Stage 36:====================================================(333 + -21) / 312]]

EP diário: 725


## Otimização Simulated Annealing

In [ ]:
##########
#Dataclass
##########

# Definição dos paramentros de controle (regras da simulação e penalidades). Padronizo as colunas das bases anteriores e faço uma limpeza dos dados de produtos que podem ter passado nos filtros anteriores.
# Gera um diagnóstico inicial para saber quais dados o otimizador está lidando antes de começar a otimização.

##################
#Modelo de Demanda
##################

#Feito inicialmente pois a informação do intercept do modelo de Roilift estava muito instavel e estava só gerando resultados negativos.
#prever_demanda - estima as vendas via elasticidade, travando o crescimento pelo estoque e pela agressividade do desconto dado.
#calc_metricas - consolida a parte financeira, calculando o faturamento, custos e a margem de cada cenário.
#func_obj - ela premia o lucro e o IP, mas aplica penalidade se o orçamento estourar.

###################
#5 Soluções Iniciais
###################

#É onde estão as regras de decisão para distribuir o desconto. 
#Testa 5 prioridades diferentes (foca em: quem é mais elástico, em quem está com IP ruim, em quem gera mais GMV, e em 2 aleatórias para testar o score final)

###################
#Simulated Annealing
###################

#gerar_vizinho - cria variações nos descontos usando mudanças aleatorias, ajustes para atingir o IP do concorrente ou troca de verba entre os produtos
#simulated_annealing - é basicamente o maestro, ele orquestra a busca global, partindo das solucoes iniciais que fizemos acima para explorar as milhões de combinações que setamos anteriormente.
#temperatura - logica de aceitar combinações mais arriscadas no começo para mapear e quando começa a esfriar ele foca no refinamento dos ganhos.
#restarts e estagnação - evita que o algoritmo fique preso em resultados medianos, forçando sempre a buscar o melhor cenário global.
#resultado - combinação ideal de rebates que maximiza o gmv e a competitividade, respeitando as regras e travas de orçamento e desconto.

###########
#RElatorio
###########

#Diagnostico completo para conseguir ter a informação, quase como um sumário.

###################
#Execução
###################

#Local onde vai alterar as informações caso mude o nome da base, e setar os parametros de otimização como orçamento de rebate, percentual de desconto, ip algo e até numero maximo de iterações que o otimizador vai rodar.
#Faz tambem um top 30 produtos, uma analise de IP e salva a otimização em XLSX ou CSV.

In [39]:
# =====================================================================
# CONFIG DO OTIMIZADOR
# =====================================================================
@dataclass
class OptConfig:
    orcamento_rebate: float
    max_desconto_pct: float
    max_mult_demanda: float = 2.0
    piso_demanda_min: float = 3.0
    ip_alvo: float = 1.0
    peso_ip: float = 5.0
    temp_inicial: float = 50000.0
    temp_final: float = 0.001
    max_iter: int = 1_000_000
    n_restarts: int = 5
    iter_por_restart: int = 0
    penalidade_orcamento: float = 100.0
    seed: Optional[int] = 42
    alpha_visitas: float = 0.2  # peso das visitas na função objetivo (0 = desligado)
 
    def __post_init__(self):
        if self.iter_por_restart == 0:
            self.iter_por_restart = self.max_iter // self.n_restarts
 
    @property
    def taxa_resfriamento(self) -> float:
        return (self.temp_final / self.temp_inicial) ** (1.0 / max(self.iter_por_restart, 1))
 
 
# =====================================================================
# PRICING PSICOLÓGICO
# =====================================================================
def aplicar_preco_psicologico(preco):
    """
    Arredonda o preço para o décimo inferior mais próximo (.00, .10, .20, até .90)
      ex: 12.87 > 12.80 | 9.43 > 9.40 
    """
    inteiro = math.floor(preco)
    decimal = round(preco - inteiro, 2)
    decimo = math.floor(decimal * 10) / 10

    return round(inteiro + decimo, 2)    
 
# =====================================================================
# PREPARAR DADOS
# =====================================================================
def preparar_dados(df_input, is_spark=True):
    rename_map = {
        "categoria": "categoria_comercial",
        "preco_por": "preco_por_atual",
        "concorrente": "Concorrente",
        "carteira": "Carteira",
        "rebate_unitario_medio": "rebate_prazo_magalu",
        "seller_id": "SellerID",
    }
 
    cols_obr = ["navigation_id", "sku", "categoria_comercial", "preco_por_atual",
                "quantidadeitens", "coef", "intercept", "SellerID", "estoque", "IP"]
    cols_opt = ["Concorrente", "Carteira", "rebate_prazo_magalu", "qtd_visitas"]
 
    if is_spark:
        for old, new in rename_map.items():
            if old in df_input.columns:
                df_input = df_input.withColumnRenamed(old, new)
        cols_usar = [c for c in cols_obr + cols_opt if c in df_input.columns]
        df = df_input.select(cols_usar).toPandas()
    else:
        df = df_input.rename(columns=rename_map)
        cols_usar = [c for c in cols_obr + cols_opt if c in df.columns]
        df = df[cols_usar].copy()
 
    df = df.dropna(subset=["coef", "preco_por_atual"])
    df = df[df["preco_por_atual"] > 0].copy()
    df["preco_por_atual"] = df["preco_por_atual"].astype(float)
    df["coef"] = df["coef"].astype(float)
    df["intercept"] = df["intercept"].fillna(0).astype(float)
    df["quantidadeitens"] = df["quantidadeitens"].fillna(0).astype(float)
    df["estoque"] = df["estoque"].fillna(9999).astype(int)
    df["IP"] = df["IP"].fillna(1.0).astype(float).replace(0.0, 1.0)
    df["preco_concorrente"] = np.where(
        df["IP"] > 0, df["preco_por_atual"] / df["IP"], df["preco_por_atual"]
    )
    for col, default in [("Concorrente", "N/A"), ("Carteira", "N/A"), ("rebate_prazo_magalu", 0.0)]:
        if col not in df.columns:
            df[col] = default
        else:
            df[col] = df[col].fillna(default)
    df["rebate_prazo_magalu"] = pd.to_numeric(df["rebate_prazo_magalu"], errors="coerce").fillna(0.0)
 
    # Normalização log-scale de visitas por categoria (peso entre 0 e 1)
    if "qtd_visitas" not in df.columns:
        df["qtd_visitas"] = 0
    df["qtd_visitas"] = df["qtd_visitas"].fillna(0).astype(float)
 
    df["peso_visitas"] = 0.0
    for cat in df["categoria_comercial"].unique():
        mask = df["categoria_comercial"] == cat
        log_v = np.log1p(df.loc[mask, "qtd_visitas"])
        v_min, v_max = log_v.min(), log_v.max()
        if v_max > v_min:
            df.loc[mask, "peso_visitas"] = (log_v - v_min) / (v_max - v_min)
        else:
            df.loc[mask, "peso_visitas"] = 0.0
 
    df = df.drop_duplicates(subset=["navigation_id"], keep="first").reset_index(drop=True)
    return df
 
 
# =====================================================================
# FILTRO DE ELEGIVEIS
# =====================================================================
def filtrar_elegiveis(df):
    mask = (df["quantidadeitens"] > 0) & (df["estoque"] > 0) & (df["preco_por_atual"] > 0)
    return df[mask].reset_index(drop=True), df[~mask].reset_index(drop=True)
 
 
# =====================================================================
# DIAGNÓSTICO
# =====================================================================
def diagnostico(df, df_ine):
    print("\n" + "="*80)
    print("  DIAGNOSTICO DOS DADOS")
    print("="*80)
    print(f"  Total: {len(df)+len(df_ine):,} | Elegiveis: {len(df):,} | Inelegiveis: {len(df_ine):,}")
    if len(df_ine) > 0:
        print(f"    Sem vendas: {(df_ine['quantidadeitens']<=0).sum():,} | Sem estoque: {(df_ine['estoque']<=0).sum():,}")
    print(f"\n  Precos: Min R${df['preco_por_atual'].min():,.2f} | Med R${df['preco_por_atual'].median():,.2f} | Max R${df['preco_por_atual'].max():,.2f}")
    print(f"  Vendas: Min {df['quantidadeitens'].min():.0f} | Med {df['quantidadeitens'].median():.0f} | Max {df['quantidadeitens'].max():.0f} | Total {df['quantidadeitens'].sum():,.0f}")
    c = df["coef"]
    print(f"  Coef: Negativo (elastico) {(c<0).sum():,} | Zero (sem efeito) {(c==0).sum():,}")
    print(f"  IP: Medio {df['IP'].mean():.4f} | >1 (nao compet.) {(df['IP']>1).sum():,} | <=1 (compet.) {(df['IP']<=1).sum():,}")
    print(f"\n  GMV atual: R$ {(df['preco_por_atual']*df['quantidadeitens']).sum():,.2f}")
    print(f"  Rebate atual (rebate_prazo_magalu): R$ {df['rebate_prazo_magalu'].sum():,.2f}")
    v_com = (df["peso_visitas"] > 0).sum()
    print(f"  SKUs com visitas: {v_com:,} | SKUs sem visitas (peso=0): {len(df)-v_com:,}")
    print("="*80)
 
 
# =====================================================================
# MODELO DE DEMANDA - CAP PROPORCIONAL
# =====================================================================
def prever_demanda(precos, rebates, qtds, coefs, estoques, max_mult=2.0, piso_min=3.0, max_desc_pct=0.10):
    r = np.maximum(rebates, 0.0)
    qtd_nova = qtds + coefs * (-r)
    desc_pct = r / np.maximum(precos, 0.01)
    fator = np.clip(desc_pct / max(max_desc_pct, 0.001), 0.0, 1.0)
    mult = 1.0 + fator * (max_mult - 1.0)
    piso = qtds + piso_min * fator
    max_qtd = np.maximum(qtds * mult, piso)
    teto = np.minimum(max_qtd, estoques.astype(float))
    return np.clip(qtd_nova, 0.0, teto)
 
 
def calc_metricas(precos, rebates, qtds, coefs, estoques, mm=2.0, pm=3.0, mdp=0.10):
    r = np.maximum(rebates, 0.0)
    po = np.maximum(precos - r, 0.01)
    qn = prever_demanda(precos, r, qtds, coefs, estoques, mm, pm, mdp)
    gmv = (po * qn).sum()
    custo = (r * qn).sum()
    return po, qn, gmv, max(custo, 0.0), gmv - max(custo, 0.0)
 
 
# =====================================================================
# FUNÇÃO OBJETIVO (com peso de visitas)
# =====================================================================
def func_obj(precos, rebates, qtds, coefs, estoques, pc, ips, pesos_visitas, cfg):
    r = np.maximum(rebates, 0.0)
    _, qn, gmv, custo, receita = calc_metricas(
        precos, r, qtds, coefs, estoques,
        cfg.max_mult_demanda, cfg.piso_demanda_min, cfg.max_desconto_pct
    )
 
    # Score base: receita líquida ponderada por visitas
    gmv_pp = np.maximum(precos - r, 0.01) * qn
    score = (gmv_pp * (1 + cfg.alpha_visitas * pesos_visitas)).sum() - custo
 
    if custo > cfg.orcamento_rebate:
        exc = custo - cfg.orcamento_rebate
        score -= cfg.penalidade_orcamento * exc + 0.1 * exc**2
 
    viol = np.maximum(r / np.maximum(precos, 0.01) - cfg.max_desconto_pct, 0.0)
    if viol.sum() > 0:
        score -= 5000.0 * viol.sum()
 
    po = np.maximum(precos - r, 0.01)
    ip_novo = po / np.maximum(pc, 0.01)
    score += cfg.peso_ip * ((ips > cfg.ip_alvo) & (ip_novo <= cfg.ip_alvo)).sum() * 100
    score -= 0.3 * r[ips <= cfg.ip_alvo * 0.85].sum()
 
    return score
 
 
# =====================================================================
# SOLUÇÕES INICIAIS
# =====================================================================
def _ok(coef):
    return coef < 0
 
def sol_greedy_ip(precos, qtds, coefs, est, ips, pc, mdp, orc, ip_alvo, mm, pm):
    n = len(precos); mr = precos * mdp; reb = np.zeros(n); rest = orc
    for i in np.argsort(-ips):
        if rest <= 0: break
        if not _ok(coefs[i]) or ips[i] <= ip_alvo * 0.9: continue
        ri = min(max(0.0, precos[i] - ip_alvo * pc[i]), mr[i])
        q = prever_demanda(precos[i:i+1], np.array([ri]), qtds[i:i+1], coefs[i:i+1], est[i:i+1], mm, pm, mdp)[0]
        c = ri * q
        if 0 < c <= rest: reb[i] = ri; rest -= c
    return reb
 
def sol_greedy_elast(precos, qtds, coefs, est, mdp, orc, mm, pm):
    n = len(precos); mr = precos * mdp; reb = np.zeros(n); rest = orc
    elast = np.abs(np.minimum(coefs, 0))
    for i in np.argsort(-elast):
        if rest <= 0 or not _ok(coefs[i]): break
        ri = mr[i]
        q = prever_demanda(precos[i:i+1], np.array([ri]), qtds[i:i+1], coefs[i:i+1], est[i:i+1], mm, pm, mdp)[0]
        c = ri * q
        if 0 < c <= rest: reb[i] = ri; rest -= c
    return reb
 
def sol_greedy_gmv(precos, qtds, coefs, est, mdp, orc, mm, pm):
    n = len(precos); mr = precos * mdp; reb = np.zeros(n); rest = orc
    gp = np.zeros(n)
    for i in range(n):
        if not _ok(coefs[i]): continue
        q = prever_demanda(precos[i:i+1], mr[i:i+1], qtds[i:i+1], coefs[i:i+1], est[i:i+1], mm, pm, mdp)[0]
        gp[i] = (precos[i] - mr[i]) * q
    for i in np.argsort(-gp):
        if rest <= 0 or not _ok(coefs[i]): break
        q = prever_demanda(precos[i:i+1], np.array([mr[i]]), qtds[i:i+1], coefs[i:i+1], est[i:i+1], mm, pm, mdp)[0]
        c = mr[i] * q
        if 0 < c <= rest: reb[i] = mr[i]; rest -= c
    return reb
 
def sol_rand(precos, coefs, mdp):
    mr = precos * mdp; mask = coefs < 0; r = np.zeros(len(precos))
    r[mask] = np.random.uniform(0, 1, mask.sum()) * mr[mask]
    return r
 
def projetar(reb, precos, qtds, coefs, est, orc, mdp, mm, pm):
    reb = np.clip(reb, 0, precos * mdp); reb[coefs >= 0] = 0.0
    _, _, _, custo, _ = calc_metricas(precos, reb, qtds, coefs, est, mm, pm, mdp)
    if custo > orc and custo > 0:
        reb *= orc / custo * 0.95
        reb = np.clip(reb, 0, precos * mdp)
        reb[coefs >= 0] = 0.0
    return reb
 
 
# =====================================================================
# SIMULATED ANNEALING
# =====================================================================
def gerar_vizinho(reb, precos, qtds, coefs, est, pc, ips, mdp, orc, ip_alvo, temp, t0, mm, pm):
    novo = reb.copy(); n = len(reb); fator = temp / t0; mr = precos * mdp
    np_ = max(1, int(min(n, max(3, n * 0.1)) * fator + 1))
    for i in random.sample(range(n), min(np_, n)):
        if mr[i] <= 0 or coefs[i] >= 0: novo[i] = 0.0; continue
        r = random.random()
        if r < 0.05: novo[i] = random.uniform(0, mr[i])
        elif r < 0.10: novo[i] = 0.0
        elif r < 0.15: novo[i] = mr[i]
        elif r < 0.25:
            rip = min(max(0.0, precos[i] - ip_alvo * pc[i]), mr[i])
            novo[i] = np.clip(rip + random.gauss(0, max(mr[i] * fator * 0.15, 0.001)), 0, mr[i])
        elif r < 0.35:
            j = random.randint(0, n - 1)
            if j != i and coefs[j] < 0:
                t = novo[i] + novo[j]; s = random.uniform(0, 1)
                novo[i] = min(s * t, mr[i]); novo[j] = min((1 - s) * t, mr[j])
        else:
            sig = mr[i] * fator * 0.3 + mr[i] * 0.02
            novo[i] = np.clip(novo[i] + random.gauss(0, max(sig, 0.001)), 0, mr[i])
    novo = np.maximum(novo, 0.0)
    return projetar(novo, precos, qtds, coefs, est, orc, mdp, mm, pm)
 
 
def simulated_annealing(df, config):
    if config.seed is not None:
        random.seed(config.seed); np.random.seed(config.seed)
 
    precos = df["preco_por_atual"].values
    qtds = df["quantidadeitens"].values
    coefs = df["coef"].values
    est = df["estoque"].values
    ips = df["IP"].values
    pc = df["preco_concorrente"].values
    pesos_visitas = df["peso_visitas"].values
    taxa = config.taxa_resfriamento
    mm = config.max_mult_demanda; pm = config.piso_demanda_min
    mdp = config.max_desconto_pct; n = len(df)
 
    sols = [
        ("Greedy IP",          sol_greedy_ip(precos, qtds, coefs, est, ips, pc, mdp, config.orcamento_rebate, config.ip_alvo, mm, pm)),
        ("Greedy Elasticidade",sol_greedy_elast(precos, qtds, coefs, est, mdp, config.orcamento_rebate, mm, pm)),
        ("Greedy GMV",         sol_greedy_gmv(precos, qtds, coefs, est, mdp, config.orcamento_rebate, mm, pm)),
        ("Aleatoria 1",        sol_rand(precos, coefs, mdp)),
        ("Aleatoria 2",        sol_rand(precos, coefs, mdp)),
    ]
    for i, (nm, s) in enumerate(sols):
        sols[i] = (nm, projetar(np.maximum(s, 0), precos, qtds, coefs, est, config.orcamento_rebate, mdp, mm, pm))
 
    print("\n" + "="*80)
    print("  SIMULATED ANNEALING")
    print("="*80)
    print(f"  Produtos: {n:,} ({(coefs<0).sum():,} elasticos, {(coefs>=0).sum():,} sem efeito)")
    print(f"  Orcamento: R$ {config.orcamento_rebate:,.2f} | Max desc: {mdp*100:.1f}%")
    print(f"  Cap: {mm:.0f}x PROPORCIONAL | Piso: {pm:.0f} un PROPORCIONAL")
    print(f"  IP alvo: {config.ip_alvo:.2f} | Alpha visitas: {config.alpha_visitas:.2f}")
    print(f"  Restarts: {config.n_restarts} | Iter/restart: {config.iter_por_restart:,}")
    print("="*80)
 
    print(f"\n  Solucoes iniciais:")
    for nm, s in sols:
        sc = func_obj(precos, s, qtds, coefs, est, pc, ips, pesos_visitas, config)
        _, _, g, c, _ = calc_metricas(precos, s, qtds, coefs, est, mm, pm, mdp)
        print(f"    {nm:<25} Score:{sc:>14,.2f} | GMV:R${g:>14,.2f} | Rebate:R${c:>10,.2f}")
 
    best_reb = None; best_sc = -float("inf"); hist = []; it_total = 0
 
    for ri in range(config.n_restarts):
        nm, si = sols[ri % len(sols)]
        cur = si.copy()
        cur_sc = func_obj(precos, cur, qtds, coefs, est, pc, ips, pesos_visitas, config)
        br = cur.copy(); br_sc = cur_sc; temp = config.temp_inicial; stag = 0
        print(f"\n  --- Restart {ri+1}/{config.n_restarts} [{nm}] ---")
        li = max(config.iter_por_restart // 5, 1)
 
        for it in range(config.iter_por_restart):
            nv = gerar_vizinho(cur, precos, qtds, coefs, est, pc, ips, mdp, config.orcamento_rebate, config.ip_alvo, temp, config.temp_inicial, mm, pm)
            nv_sc = func_obj(precos, nv, qtds, coefs, est, pc, ips, pesos_visitas, config)
            d = nv_sc - cur_sc
            if d > 0 or random.random() < math.exp(d / max(temp, 1e-10)):
                cur = nv; cur_sc = nv_sc
                if cur_sc > br_sc: br_sc = cur_sc; br = cur.copy(); stag = 0
                else: stag += 1
            else:
                stag += 1
            temp *= taxa; it_total += 1
 
            if it % li == 0:
                _, _, g, c, r = calc_metricas(precos, br, qtds, coefs, est, mm, pm, mdp)
                pct = it / config.iter_por_restart * 100
                hist.append({"restart": ri, "iter": it_total, "temp": temp, "score": br_sc, "gmv": g, "rebate": c})
                print(f"    [{pct:5.1f}%] Score{br_sc:>14,.2f} | GMV R${g:>14,.2f} | Rebate R${c:>10,.2f} | Temp{temp:>10.2f}")
 
            if stag > config.iter_por_restart // 10:
                sr = sol_rand(precos, coefs, mdp)
                sr = projetar(np.maximum(sr, 0), precos, qtds, coefs, est, config.orcamento_rebate, mdp, mm, pm)
                cur = 0.7 * br + 0.3 * sr
                cur = np.maximum(cur, 0)
                cur = projetar(cur, precos, qtds, coefs, est, config.orcamento_rebate, mdp, mm, pm)
                cur_sc = func_obj(precos, cur, qtds, coefs, est, pc, ips, pesos_visitas, config)
                stag = 0
 
        if br_sc > best_sc: best_sc = br_sc; best_reb = br.copy()
        _, _, g, c, _ = calc_metricas(precos, br, qtds, coefs, est, mm, pm, mdp)
        print(f"  Restart {ri+1} fim: Score{br_sc:>14,.2f} | GMV R${g:>14,.2f} | Rebate R${c:>10,.2f}")
 
    print("\n" + "="*80)
    _, _, gf, cf, rf = calc_metricas(precos, best_reb, qtds, coefs, est, mm, pm, mdp)
    print(f"  MELHOR: Score{best_sc:>14,.2f} | GMV:R${gf:>14,.2f} | Rebate:R${cf:>10,.2f} | Receita:R${rf:>14,.2f}")
    print("="*80)
    return best_reb, hist
 
 
# =====================================================================
# RELATÓRIO
# =====================================================================
def gerar_relatorio(df, df_ine, rebates, config):
    precos = df["preco_por_atual"].values
    qtds = df["quantidadeitens"].values
    coefs = df["coef"].values
    est = df["estoque"].values
    ips = df["IP"].values
    pc = df["preco_concorrente"].values
    reb_atual = df["rebate_prazo_magalu"].values
    rebates = np.maximum(rebates, 0.0)
    mm = config.max_mult_demanda; pm = config.piso_demanda_min; mdp = config.max_desconto_pct
 
    po, qc, gmv_t, custo_t, receita_t = calc_metricas(precos, rebates, qtds, coefs, est, mm, pm, mdp)
    _, qs, gmv_s, _, _ = calc_metricas(precos, np.zeros(len(df)), qtds, coefs, est, mm, pm, mdp)
 
    # Aplicar pricing psicológico no preco_otimizado
    po_psico = np.array([aplicar_preco_psicologico(p) for p in po])
 
    # Recalcular rebate para ser consistente com o preço psicológico
    rebates_psico = np.maximum(precos - po_psico, 0.0)
 
    ip_novo = po_psico / np.maximum(pc, 0.01)
    custo_pp = np.maximum(rebates_psico * qc, 0.0)
    gmv_pp_com = po_psico * qc
    gmv_pp_sem = precos * qs
 
    resultado = pd.DataFrame({
        "navigation_id":       df["navigation_id"].values,
        "sku":                 df["sku"].values,
        "categoria_comercial": df["categoria_comercial"].values,
        "seller_id":           df["SellerID"].values,
        "concorrente":         df["Concorrente"].values if "Concorrente" in df.columns else "N/A",
        "carteira":            df["Carteira"].values if "Carteira" in df.columns else "N/A",
        "preco_atual":         precos.round(2),
        "preco_concorrente":   pc.round(2),
        "IP_atual":            ips.round(4),
        "rebate_atual_magalu": reb_atual.round(2),
        "rebate_otimizado":    rebates_psico.round(2),
        "desconto_pct":        ((rebates_psico / np.maximum(precos, 0.01)) * 100).round(2),
        "preco_otimizado":     po_psico.round(2),
        "IP_novo":             ip_novo.round(4),
        "delta_IP":            (ip_novo - ips).round(4),
        "status_ip":           np.where(ip_novo <= config.ip_alvo, "COMPETITIVO",
                                   np.where(ip_novo <= config.ip_alvo * 1.05, "MARGINAL", "NAO_COMPETITIVO")),
        "qtd_atual":           qtds.round(0),
        "qtd_prevista":        qc.round(1),
        "delta_qtd":           (qc - qtds).round(1),
        "mult_demanda":        np.where(qtds > 0, (qc / qtds).round(2), 0.0),
        "gmv_atual":           gmv_pp_sem.round(2),
        "gmv_otimizado":       gmv_pp_com.round(2),
        "delta_gmv":           (gmv_pp_com - gmv_pp_sem).round(2),
        "custo_rebate":        custo_pp.round(2),
        "lucro_previsto":      (gmv_pp_com - custo_pp).round(2),
        "roi_rebate":          np.where(custo_pp > 0.01, np.round((gmv_pp_com - gmv_pp_sem) / np.maximum(custo_pp, 0.01), 2), 0.0),
        "qtd_visitas":         df["qtd_visitas"].values,
        "peso_visitas":        df["peso_visitas"].values.round(4),
        "estoque":             est,
    })
    resultado = resultado.sort_values("gmv_otimizado", ascending=False).reset_index(drop=True)
 
    nr = (rebates_psico > 0.01).sum()
    mk = resultado["rebate_otimizado"] > 0.01
    dm = resultado.loc[mk, "desconto_pct"].mean() if mk.any() else 0.0
    ipa = (ips <= config.ip_alvo).sum()
    ipd = (ip_novo <= config.ip_alvo).sum()
    rat = reb_atual.sum()
 
    print("\n" + "="*80)
    print("  SUMARIO FINAL DA OTIMIZACAO")
    print("="*80)
    print(f"  {'Produtos elegiveis:':<40} {len(df):>10,}")
    print(f"  {'Produtos inelegiveis:':<40} {len(df_ine):>10,}")
    print(f"  {'Produtos com rebate > 0:':<40} {nr:>10,}")
    print(f"  {'Produtos com coef < 0 (elasticos):':<40} {(coefs<0).sum():>10,}")
    print("-"*80)
    print(f"  {'GMV ATUAL:':<40} R$ {gmv_s:>14,.2f}")
    print(f"  {'GMV OTIMIZADO:':<40} R$ {gmv_t:>14,.2f}")
    print(f"  {'DELTA GMV (+):':<40} R$ {gmv_t-gmv_s:>14,.2f}")
    if gmv_s > 0: print(f"  {'Crescimento GMV:':<40} {(gmv_t-gmv_s)/gmv_s*100:>13.2f}%")
    print("-"*80)
    print(f"  {'Rebate ATUAL (rebate_prazo_magalu):':<40} R$ {rat:>14,.2f}")
    print(f"  {'Rebate OTIMIZADO (novo):':<40} R$ {custo_t:>14,.2f}")
    print(f"  {'Orcamento disponivel:':<40} R$ {config.orcamento_rebate:>14,.2f}")
    print(f"  {'Orcamento utilizado:':<40} {custo_t/config.orcamento_rebate*100:>13.2f}%")
    print("-"*80)
    print(f"  {'Receita Liquida (GMV-Rebate):':<40} R$ {receita_t:>14,.2f}")
    print(f"  {'Desconto Medio (c/ rebate):':<40} {dm:>13.2f}%")
    if custo_t > 0: print(f"  {'ROI do Rebate (deltaGMV/custo):':<40} {(gmv_t-gmv_s)/custo_t:>13.2f}x")
    print("-"*80)
    print(f"  {'IP:':<40}")
    print(f"  {'  Competitivos ANTES:':<40} {ipa:>10,}")
    print(f"  {'  Competitivos DEPOIS:':<40} {ipd:>10,}")
    print(f"  {'  IPs corrigidos:':<40} {ipd-ipa:>10,}")
    print(f"  {'  IP medio antes:':<40} {ips.mean():>13.4f}")
    print(f"  {'  IP medio depois:':<40} {ip_novo.mean():>13.4f}")
    print("="*80)
    return resultado
 
 
# =====================================================================
# ENTRY POINT
# =====================================================================
def otimizar_rebate(
    df_input,
    orcamento_rebate,
    max_desconto_pct=0.10,
    max_mult_demanda=2.0,
    piso_demanda_min=3.0,
    ip_alvo=1.0,
    max_iter=1_000_000,
    n_restarts=5,
    seed=42,
    alpha_visitas=0.2,
    is_spark=True,
):
    config = OptConfig(
        orcamento_rebate=orcamento_rebate,
        max_desconto_pct=max_desconto_pct,
        max_mult_demanda=max_mult_demanda,
        piso_demanda_min=piso_demanda_min,
        ip_alvo=ip_alvo,
        max_iter=max_iter,
        n_restarts=n_restarts,
        seed=seed,
        alpha_visitas=alpha_visitas,
    )
    df = preparar_dados(df_input, is_spark=is_spark)
    df_el, df_ine = filtrar_elegiveis(df)
    diagnostico(df_el, df_ine)
    if len(df_el) == 0:
        print("\n  ERRO: Nenhum produto elegivel!")
        return pd.DataFrame(), []
    rebates, hist = simulated_annealing(df_el, config)
    resultado = gerar_relatorio(df_el, df_ine, rebates, config)
    return resultado, hist
 
 
# =====================================================================
# EXECUÇÃO
# =====================================================================
resultado_ud_mensal, hist_ud_m = otimizar_rebate(
    df_input=df_ep_mensal,
    orcamento_rebate=800_000,
    max_desconto_pct=0.10,
    max_mult_demanda=2.0,
    piso_demanda_min=3.0,
    ip_alvo=0.98,
    max_iter=1_000_000,
    n_restarts=5,
    seed=42,
    alpha_visitas=0.2,
    is_spark=True,
)
 
print("\n  TOP 30 PRODUTOS POR GMV OTIMIZADO:")
print("-"*190)
cols = [
    "navigation_id", "sku", "categoria_comercial",
    "preco_atual", "preco_concorrente", "IP_atual",
    "rebate_otimizado", "desconto_pct", "preco_otimizado",
    "IP_novo", "status_ip",
    "qtd_atual", "qtd_prevista", "mult_demanda",
    "gmv_otimizado", "custo_rebate", "roi_rebate",
    "qtd_visitas", "peso_visitas",
]
print(resultado_ud_mensal[cols].head(30).to_string(index=False, float_format="%.2f"))
 
print("\n\n  SUMARIO POR CATEGORIA:")
print("-"*140)
sumario = resultado_ud_mensal.groupby("categoria_comercial").agg(
    n_produtos=("navigation_id", "count"),
    rebate_total=("custo_rebate", "sum"),
    gmv_atual=("gmv_atual", "sum"),
    gmv_otimizado=("gmv_otimizado", "sum"),
    delta_gmv=("delta_gmv", "sum"),
    desc_medio=("desconto_pct", "mean"),
    mult_medio=("mult_demanda", "mean"),
    ip_medio_novo=("IP_novo", "mean"),
    n_competitivos=("status_ip", lambda x: (x == "COMPETITIVO").sum()),
).round(2)
print(sumario.to_string())
 
print("\n\n  ANALISE DE IP POR STATUS:")
print("-"*70)
ip_a = resultado_ud_mensal.groupby("status_ip").agg(
    n_produtos=("navigation_id", "count"),
    ip_medio=("IP_novo", "mean"),
    rebate_medio=("rebate_otimizado", "mean"),
    gmv_total=("gmv_otimizado", "sum"),
).round(2)
print(ip_a.to_string())

                                                                                ]]]]]


  DIAGNOSTICO DOS DADOS
  Total: 725 | Elegiveis: 725 | Inelegiveis: 0

  Precos: Min R$7.50 | Med R$154.44 | Max R$6,064.99
  Vendas: Min 1 | Med 1 | Max 42 | Total 1,514
  Coef: Negativo (elastico) 651 | Zero (sem efeito) 74
  IP: Medio 1.0211 | >1 (nao compet.) 158 | <=1 (compet.) 567

  GMV atual: R$ 429,907.01
  Rebate atual (rebate_prazo_magalu): R$ 6,066.17
  SKUs com visitas: 721 | SKUs sem visitas (peso=0): 4

  SIMULATED ANNEALING
  Produtos: 725 (651 elasticos, 74 sem efeito)
  Orcamento: R$ 800,000.00 | Max desc: 10.0%
  Cap: 2x PROPORCIONAL | Piso: 3 un PROPORCIONAL
  IP alvo: 0.98 | Alpha visitas: 0.20
  Restarts: 5 | Iter/restart: 200,000

  Solucoes iniciais:
    Greedy IP                 Score:    850,051.34 | GMV:R$    600,198.39 | Rebate:R$ 31,454.47
    Greedy Elasticidade       Score:  1,110,190.94 | GMV:R$    855,966.08 | Rebate:R$ 90,973.89
    Greedy GMV                Score:  1,110,190.94 | GMV:R$    855,966.08 | Rebate:R$ 90,973.89
    Aleatoria 1            

[Stage 36:====================================================(333 + -21) / 312]

    [ 20.0%] Score    879,554.30 | GMV R$    674,859.11 | Rebate R$ 44,539.17 | Temp   1442.57
    [ 40.0%] Score  1,098,762.83 | GMV R$    832,491.46 | Rebate R$ 75,323.84 | Temp     41.62
    [ 60.0%] Score  1,122,433.29 | GMV R$    859,803.49 | Rebate R$ 82,644.84 | Temp      1.20
    [ 80.0%] Score  1,123,153.44 | GMV R$    860,768.27 | Rebate R$ 82,969.13 | Temp      0.03
  Restart 1 fim: Score  1,123,160.25 | GMV R$    860,787.43 | Rebate R$ 82,983.58

  --- Restart 2/5 [Greedy Elasticidade] ---
    [  0.0%] Score  1,110,190.94 | GMV R$    855,966.08 | Rebate R$ 90,973.89 | Temp  49995.57


[Stage 36:====================================================(333 + -21) / 312]

...

[Stage 36:====================================================(333 + -21) / 312]

    [ 20.0%] Score    887,624.17 | GMV R$    674,102.15 | Rebate R$ 44,869.32 | Temp   1442.57
    [ 40.0%] Score  1,096,146.37 | GMV R$    828,466.86 | Rebate R$ 74,695.99 | Temp     41.62
    [ 60.0%] Score  1,122,227.11 | GMV R$    859,617.75 | Rebate R$ 82,649.13 | Temp      1.20
    [ 80.0%] Score  1,123,156.12 | GMV R$    860,789.88 | Rebate R$ 82,990.48 | Temp      0.03
  Restart 4 fim: Score  1,123,162.59 | GMV R$    860,787.84 | Rebate R$ 82,981.65

  --- Restart 5/5 [Aleatoria 2] ---
    [  0.0%] Score    865,817.45 | GMV R$    633,374.98 | Rebate R$ 36,051.04 | Temp  49995.57


[Stage 36:====================================================(333 + -21) / 312]

    [ 20.0%] Score    894,656.18 | GMV R$    642,781.57 | Rebate R$ 35,349.35 | Temp   1442.57
    [ 40.0%] Score  1,094,458.21 | GMV R$    827,605.95 | Rebate R$ 74,427.88 | Temp     41.62
    [ 60.0%] Score  1,122,692.61 | GMV R$    860,236.81 | Rebate R$ 82,869.95 | Temp      1.20
    [ 80.0%] Score  1,123,156.96 | GMV R$    860,792.96 | Rebate R$ 82,993.16 | Temp      0.03
  Restart 5 fim: Score  1,123,162.96 | GMV R$    860,787.01 | Rebate R$ 82,980.33

  MELHOR: Score  1,123,163.14 | GMV:R$    860,786.33 | Rebate:R$ 82,979.36 | Receita:R$    777,806.96

  SUMARIO FINAL DA OTIMIZACAO
  Produtos elegiveis:                             725
  Produtos inelegiveis:                             0
  Produtos com rebate > 0:                        661
  Produtos com coef < 0 (elasticos):              651
--------------------------------------------------------------------------------
  GMV ATUAL:                               R$     417,794.84
  GMV OTIMIZADO:                           R$ 

[Stage 36:====================================================(333 + -21) / 312]

## Testes

In [44]:
@dataclass
class OptConfig:
    orcamento_rebate: float
    max_desconto_pct: float
    max_mult_demanda: float = 2.0
    piso_demanda_min: float = 3.0
    ip_alvo: float = 1.0
    peso_ip: float = 5.0
    temp_inicial: float = 50000.0
    temp_final: float = 0.001
    max_iter: int = 1_000_000
    n_restarts: int = 5
    iter_por_restart: int = 0
    penalidade_orcamento: float = 100.0
    seed: Optional[int] = 42
    def __post_init__(self):
        if self.iter_por_restart == 0:
            self.iter_por_restart = self.max_iter // self.n_restarts
    @property
    def taxa_resfriamento(self) -> float:
        return (self.temp_final / self.temp_inicial) ** (1.0 / max(self.iter_por_restart, 1))

def preparar_dados(df_input, is_spark=True):
    # Mapeamento de nomes de campos: nome atual > nome que o otimizador usa internamente
    rename_map = {
        "categoria": "categoria_comercial",
        "preco_por": "preco_por_atual",
        "concorrente": "Concorrente",
        "carteira": "Carteira",
        "rebate_unitario_medio": "rebate_prazo_magalu",
        "seller_id": "SellerID",
    }
    
    cols_obr = ["navigation_id", "sku", "categoria_comercial", "preco_por_atual",
                "quantidadeitens", "coef", "intercept", "SellerID", "estoque", "IP"]
    cols_opt = ["Concorrente", "Carteira", "rebate_prazo_magalu"]

    if is_spark:
        # Renomear colunas antes de converter
        for old, new in rename_map.items():
            if old in df_input.columns:
                df_input = df_input.withColumnRenamed(old, new)
        cols_usar = [c for c in cols_obr + cols_opt if c in df_input.columns]
        df = df_input.select(cols_usar).toPandas()
    else:
        df = df_input.rename(columns=rename_map)
        cols_usar = [c for c in cols_obr + cols_opt if c in df.columns]
        df = df[cols_usar].copy()

    df = df.dropna(subset=["coef", "preco_por_atual"])
    df = df[df["preco_por_atual"] > 0].copy()
    df["preco_por_atual"] = df["preco_por_atual"].astype(float)
    df["coef"] = df["coef"].astype(float)
    df["intercept"] = df["intercept"].fillna(0).astype(float)
    df["quantidadeitens"] = df["quantidadeitens"].fillna(0).astype(float)
    df["estoque"] = df["estoque"].fillna(9999).astype(int)
    df["IP"] = df["IP"].fillna(1.0).astype(float).replace(0.0, 1.0)
    df["preco_concorrente"] = np.where(
        df["IP"] > 0, df["preco_por_atual"] / df["IP"], df["preco_por_atual"]
    )
    for col, default in [("Concorrente", "N/A"), ("Carteira", "N/A"), ("rebate_prazo_magalu", 0.0)]:
        if col not in df.columns:
            df[col] = default
        else:
            df[col] = df[col].fillna(default)
    df["rebate_prazo_magalu"] = pd.to_numeric(df["rebate_prazo_magalu"], errors="coerce").fillna(0.0)
    df = df.drop_duplicates(subset=["navigation_id"], keep="first").reset_index(drop=True)
    return df

def filtrar_elegiveis(df):
    mask = (df["quantidadeitens"]>0) & (df["estoque"]>0) & (df["preco_por_atual"]>0)
    return df[mask].reset_index(drop=True), df[~mask].reset_index(drop=True)

def diagnostico(df, df_ine):
    print("\n" + "="*80)
    print("  DIAGNOSTICO DOS DADOS")
    print("="*80)
    print(f"  Total: {len(df)+len(df_ine):,} | Elegiveis: {len(df):,} | Inelegiveis: {len(df_ine):,}")
    if len(df_ine)>0:
        print(f"    Sem vendas: {(df_ine['quantidadeitens']<=0).sum():,} | Sem estoque: {(df_ine['estoque']<=0).sum():,}")
    print(f"\n  Precos: Min R${df['preco_por_atual'].min():,.2f} | Med R${df['preco_por_atual'].median():,.2f} | Max R${df['preco_por_atual'].max():,.2f}")
    print(f"  Vendas: Min {df['quantidadeitens'].min():.0f} | Med {df['quantidadeitens'].median():.0f} | Max {df['quantidadeitens'].max():.0f} | Total {df['quantidadeitens'].sum():,.0f}")
    c = df["coef"]
    print(f"  Coef: Negativo (elastico) {(c<0).sum():,} | Zero (sem efeito) {(c==0).sum():,}")
    print(f"  IP: Medio {df['IP'].mean():.4f} | >1 (nao compet.) {(df['IP']>1).sum():,} | <=1 (compet.) {(df['IP']<=1).sum():,}")
    print(f"\n  GMV atual: R$ {(df['preco_por_atual']*df['quantidadeitens']).sum():,.2f}")
    print(f"  Rebate atual (rebate_prazo_magalu): R$ {df['rebate_prazo_magalu'].sum():,.2f}")
    print("="*80)

# =====================================================================
# MODELO DE DEMANDA - CAP PROPORCIONAL
# =====================================================================
def prever_demanda(precos, rebates, qtds, coefs, estoques, max_mult=2.0, piso_min=3.0, max_desc_pct=0.10):
    """
    Q_nova = qtd_atual - coef * rebate  (coef negativo -> desconto aumenta demanda)

    Cap PROPORCIONAL ao desconto:
      fator = desconto_dado / desconto_maximo  (0 a 1)
      mult_max = 1 + fator * (max_mult - 1)
      piso = qtd + piso_min * fator

    Desc 10% -> mult 2.0x | Desc 5% -> mult 1.5x | Desc 1% -> mult 1.1x | Desc 0.2% -> mult 1.02x
    """
    r = np.maximum(rebates, 0.0)
    qtd_nova = qtds + coefs * (-r)
    desc_pct = r / np.maximum(precos, 0.01)
    fator = np.clip(desc_pct / max(max_desc_pct, 0.001), 0.0, 1.0)
    mult = 1.0 + fator * (max_mult - 1.0)
    piso = qtds + piso_min * fator
    max_qtd = np.maximum(qtds * mult, piso)
    teto = np.minimum(max_qtd, estoques.astype(float))
    return np.clip(qtd_nova, 0.0, teto)

def calc_metricas(precos, rebates, qtds, coefs, estoques, mm=2.0, pm=3.0, mdp=0.10):
    r = np.maximum(rebates, 0.0)
    po = np.maximum(precos - r, 0.01)
    qn = prever_demanda(precos, r, qtds, coefs, estoques, mm, pm, mdp)
    gmv = (po * qn).sum()
    custo = (r * qn).sum()
    return po, qn, gmv, max(custo, 0.0), gmv - max(custo, 0.0)

def func_obj(precos, rebates, qtds, coefs, estoques, pc, ips, cfg):
    r = np.maximum(rebates, 0.0)
    _, _, gmv, custo, receita = calc_metricas(precos, r, qtds, coefs, estoques, cfg.max_mult_demanda, cfg.piso_demanda_min, cfg.max_desconto_pct)
    score = receita
    if custo > cfg.orcamento_rebate:
        exc = custo - cfg.orcamento_rebate
        score -= cfg.penalidade_orcamento * exc + 0.1 * exc**2
    viol = np.maximum(r/np.maximum(precos,0.01) - cfg.max_desconto_pct, 0.0)
    if viol.sum() > 0: score -= 5000.0 * viol.sum()
    po = np.maximum(precos - r, 0.01)
    ip_novo = po / np.maximum(pc, 0.01)
    score += cfg.peso_ip * ((ips > cfg.ip_alvo) & (ip_novo <= cfg.ip_alvo)).sum() * 100
    score -= 0.3 * r[ips <= cfg.ip_alvo * 0.85].sum()
    return score

# =====================================================================
# SOLUCOES INICIAIS
# =====================================================================
def _ok(coef): return coef < 0

def sol_greedy_ip(precos, qtds, coefs, est, ips, pc, mdp, orc, ip_alvo, mm, pm):
    n = len(precos); mr = precos*mdp; reb = np.zeros(n); rest = orc
    for i in np.argsort(-ips):
        if rest <= 0: break
        if not _ok(coefs[i]) or ips[i] <= ip_alvo*0.9: continue
        ri = min(max(0.0, precos[i]-ip_alvo*pc[i]), mr[i])
        q = prever_demanda(precos[i:i+1], np.array([ri]), qtds[i:i+1], coefs[i:i+1], est[i:i+1], mm, pm, mdp)[0]
        c = ri*q
        if 0 < c <= rest: reb[i]=ri; rest-=c
    return reb

def sol_greedy_elast(precos, qtds, coefs, est, mdp, orc, mm, pm):
    n = len(precos); mr = precos*mdp; reb = np.zeros(n); rest = orc
    elast = np.abs(np.minimum(coefs, 0))
    for i in np.argsort(-elast):
        if rest <= 0 or not _ok(coefs[i]): break
        ri = mr[i]
        q = prever_demanda(precos[i:i+1], np.array([ri]), qtds[i:i+1], coefs[i:i+1], est[i:i+1], mm, pm, mdp)[0]
        c = ri*q
        if 0 < c <= rest: reb[i]=ri; rest-=c
    return reb

def sol_greedy_gmv(precos, qtds, coefs, est, mdp, orc, mm, pm):
    n = len(precos); mr = precos*mdp; reb = np.zeros(n); rest = orc
    gp = np.zeros(n)
    for i in range(n):
        if not _ok(coefs[i]): continue
        q = prever_demanda(precos[i:i+1], mr[i:i+1], qtds[i:i+1], coefs[i:i+1], est[i:i+1], mm, pm, mdp)[0]
        gp[i] = (precos[i]-mr[i])*q
    for i in np.argsort(-gp):
        if rest <= 0 or not _ok(coefs[i]): break
        q = prever_demanda(precos[i:i+1], np.array([mr[i]]), qtds[i:i+1], coefs[i:i+1], est[i:i+1], mm, pm, mdp)[0]
        c = mr[i]*q
        if 0 < c <= rest: reb[i]=mr[i]; rest-=c
    return reb

def sol_rand(precos, coefs, mdp):
    mr = precos*mdp; mask = coefs<0; r = np.zeros(len(precos))
    r[mask] = np.random.uniform(0,1,mask.sum())*mr[mask]
    return r

def projetar(reb, precos, qtds, coefs, est, orc, mdp, mm, pm):
    reb = np.clip(reb, 0, precos*mdp); reb[coefs>=0]=0.0
    _, _, _, custo, _ = calc_metricas(precos, reb, qtds, coefs, est, mm, pm, mdp)
    if custo > orc and custo > 0:
        reb *= orc/custo*0.95; reb = np.clip(reb, 0, precos*mdp); reb[coefs>=0]=0.0
    return reb

# =====================================================================
# SIMULATED ANNEALING
# =====================================================================
def gerar_vizinho(reb, precos, qtds, coefs, est, pc, ips, mdp, orc, ip_alvo, temp, t0, mm, pm):
    novo = reb.copy(); n = len(reb); fator = temp/t0; mr = precos*mdp
    np_ = max(1, int(min(n, max(3, n*0.1))*fator+1))
    for i in random.sample(range(n), min(np_, n)):
        if mr[i]<=0 or coefs[i]>=0: novo[i]=0.0; continue
        r = random.random()
        if r<0.05: novo[i]=random.uniform(0,mr[i])
        elif r<0.10: novo[i]=0.0
        elif r<0.15: novo[i]=mr[i]
        elif r<0.25:
            rip = min(max(0.0, precos[i]-ip_alvo*pc[i]), mr[i])
            novo[i]=np.clip(rip+random.gauss(0,max(mr[i]*fator*0.15,0.001)),0,mr[i])
        elif r<0.35:
            j = random.randint(0,n-1)
            if j!=i and coefs[j]<0:
                t=novo[i]+novo[j]; s=random.uniform(0,1)
                novo[i]=min(s*t,mr[i]); novo[j]=min((1-s)*t,mr[j])
        else:
            sig = mr[i]*fator*0.3+mr[i]*0.02
            novo[i]=np.clip(novo[i]+random.gauss(0,max(sig,0.001)),0,mr[i])
    novo = np.maximum(novo, 0.0)
    return projetar(novo, precos, qtds, coefs, est, orc, mdp, mm, pm)

def simulated_annealing(df, config):
    if config.seed is not None: random.seed(config.seed); np.random.seed(config.seed)
    precos=df["preco_por_atual"].values; qtds=df["quantidadeitens"].values
    coefs=df["coef"].values; est=df["estoque"].values
    ips=df["IP"].values; pc=df["preco_concorrente"].values
    taxa=config.taxa_resfriamento; mm=config.max_mult_demanda; pm=config.piso_demanda_min; mdp=config.max_desconto_pct; n=len(df)

    sols = [
        ("Greedy IP", sol_greedy_ip(precos,qtds,coefs,est,ips,pc,mdp,config.orcamento_rebate,config.ip_alvo,mm,pm)),
        ("Greedy Elasticidade", sol_greedy_elast(precos,qtds,coefs,est,mdp,config.orcamento_rebate,mm,pm)),
        ("Greedy GMV", sol_greedy_gmv(precos,qtds,coefs,est,mdp,config.orcamento_rebate,mm,pm)),
        ("Aleatoria 1", sol_rand(precos,coefs,mdp)),
        ("Aleatoria 2", sol_rand(precos,coefs,mdp)),
    ]
    for i,(nm,s) in enumerate(sols):
        sols[i] = (nm, projetar(np.maximum(s,0), precos, qtds, coefs, est, config.orcamento_rebate, mdp, mm, pm))

    print("\n"+"="*80)
    print("  SIMULATED ANNEALING ")
    print("="*80)
    print(f"  Produtos: {n:,} ({(coefs<0).sum():,} elasticos, {(coefs>=0).sum():,} sem efeito)")
    print(f"  Orcamento: R$ {config.orcamento_rebate:,.2f} | Max desc: {mdp*100:.1f}%")
    print(f"  Cap: {mm:.0f}x PROPORCIONAL | Piso: {pm:.0f} un PROPORCIONAL")
    print(f"  IP alvo: {config.ip_alvo:.2f} | Restarts: {config.n_restarts} | Iter/restart: {config.iter_por_restart:,}")
    print("="*80)

    print(f"\n  Solucoes iniciais:")
    for nm,s in sols:
        sc=func_obj(precos,s,qtds,coefs,est,pc,ips,config)
        _,_,g,c,_=calc_metricas(precos,s,qtds,coefs,est,mm,pm,mdp)
        print(f"    {nm:<25} Score:{sc:>14,.2f} | GMV:R${g:>14,.2f} | Rebate:R${c:>10,.2f}")

    best_reb=None; best_sc=-float("inf"); hist=[]; it_total=0

    for ri in range(config.n_restarts):
        nm,si = sols[ri % len(sols)]
        cur=si.copy(); cur_sc=func_obj(precos,cur,qtds,coefs,est,pc,ips,config)
        br=cur.copy(); br_sc=cur_sc; temp=config.temp_inicial; stag=0
        print(f"\n  --- Restart {ri+1}/{config.n_restarts} [{nm}] ---")
        li = max(config.iter_por_restart//5, 1)

        for it in range(config.iter_por_restart):
            nv = gerar_vizinho(cur,precos,qtds,coefs,est,pc,ips,mdp,config.orcamento_rebate,config.ip_alvo,temp,config.temp_inicial,mm,pm)
            nv_sc = func_obj(precos,nv,qtds,coefs,est,pc,ips,config)
            d = nv_sc - cur_sc
            if d>0 or random.random()<math.exp(d/max(temp,1e-10)):
                cur=nv; cur_sc=nv_sc
                if cur_sc>br_sc: br_sc=cur_sc; br=cur.copy(); stag=0
                else: stag+=1
            else: stag+=1
            temp*=taxa; it_total+=1

            if it%li==0:
                _,_,g,c,r=calc_metricas(precos,br,qtds,coefs,est,mm,pm,mdp)
                pct=it/config.iter_por_restart*100
                hist.append({"restart":ri,"iter":it_total,"temp":temp,"score":br_sc,"gmv":g,"rebate":c})
                print(f"    [{pct:5.1f}%] Score{br_sc:>14,.2f} | GMV R${g:>14,.2f} | Rebate R${c:>10,.2f} | Temp{temp:>10.2f}")

            if stag > config.iter_por_restart//10:
                sr=sol_rand(precos,coefs,mdp)
                sr=projetar(np.maximum(sr,0),precos,qtds,coefs,est,config.orcamento_rebate,mdp,mm,pm)
                cur=0.7*br+0.3*sr; cur=np.maximum(cur,0)
                cur=projetar(cur,precos,qtds,coefs,est,config.orcamento_rebate,mdp,mm,pm)
                cur_sc=func_obj(precos,cur,qtds,coefs,est,pc,ips,config); stag=0

        if br_sc>best_sc: best_sc=br_sc; best_reb=br.copy()
        _,_,g,c,_=calc_metricas(precos,br,qtds,coefs,est,mm,pm,mdp)
        print(f"  Restart {ri+1} fim: Score{br_sc:>14,.2f} | GMV R${g:>14,.2f} | Rebate R${c:>10,.2f}")

    print("\n"+"="*80)
    _,_,gf,cf,rf=calc_metricas(precos,best_reb,qtds,coefs,est,mm,pm,mdp)
    print(f"  MELHOR: Score{best_sc:>14,.2f} | GMV:R${gf:>14,.2f} | Rebate:R${cf:>10,.2f} | Receita:R${rf:>14,.2f}")
    print("="*80)
    return best_reb, hist

# =====================================================================
# RELATORIO
# =====================================================================
def gerar_relatorio(df, df_ine, rebates, config):
    precos=df["preco_por_atual"].values; qtds=df["quantidadeitens"].values
    coefs=df["coef"].values; est=df["estoque"].values
    ips=df["IP"].values; pc=df["preco_concorrente"].values; reb_atual=df["rebate_prazo_magalu"].values
    rebates=np.maximum(rebates,0.0); mm=config.max_mult_demanda; pm=config.piso_demanda_min; mdp=config.max_desconto_pct

    po,qc,gmv_t,custo_t,receita_t = calc_metricas(precos,rebates,qtds,coefs,est,mm,pm,mdp)
    _,qs,gmv_s,_,_ = calc_metricas(precos,np.zeros(len(df)),qtds,coefs,est,mm,pm,mdp)

    ip_novo = po/np.maximum(pc,0.01)
    custo_pp = np.maximum(rebates*qc, 0.0)
    gmv_pp_com = po*qc; gmv_pp_sem = precos*qs

    resultado = pd.DataFrame({
        "navigation_id": df["navigation_id"].values,
        "sku": df["sku"].values,
        "categoria_comercial": df["categoria_comercial"].values,
        "seller_id": df["SellerID"].values,
        "concorrente": df["Concorrente"].values if "Concorrente" in df.columns else "N/A",
        "carteira": df["Carteira"].values if "Carteira" in df.columns else "N/A",
        "preco_atual": precos.round(2),
        "preco_concorrente": pc.round(2),
        "IP_atual": ips.round(4),
        "rebate_atual_magalu": reb_atual.round(2),
        "rebate_otimizado": rebates.round(2),
        "desconto_pct": ((rebates/np.maximum(precos,0.01))*100).round(2),
        "preco_otimizado": po.round(2),
        "IP_novo": ip_novo.round(4),
        "delta_IP": (ip_novo - ips).round(4),
        "status_ip": np.where(ip_novo<=config.ip_alvo,"COMPETITIVO",np.where(ip_novo<=config.ip_alvo*1.05,"MARGINAL","NAO_COMPETITIVO")),
        "qtd_atual": qtds.round(0),
        "qtd_prevista": qc.round(1),
        "delta_qtd": (qc-qtds).round(1),
        "mult_demanda": np.where(qtds>0, (qc/qtds).round(2), 0.0),
        "gmv_atual": gmv_pp_sem.round(2),
        "gmv_otimizado": gmv_pp_com.round(2),
        "delta_gmv": (gmv_pp_com-gmv_pp_sem).round(2),
        "custo_rebate": custo_pp.round(2),
        "lucro_previsto": (gmv_pp_com-custo_pp).round(2),
        "roi_rebate": np.where(custo_pp>0.01, np.round((gmv_pp_com-gmv_pp_sem)/np.maximum(custo_pp,0.01),2), 0.0),
        "estoque": est,
    })
    resultado = resultado.sort_values("gmv_otimizado", ascending=False).reset_index(drop=True)

    nr = (rebates>0.01).sum()
    mk = resultado["rebate_otimizado"]>0.01
    dm = resultado.loc[mk,"desconto_pct"].mean() if mk.any() else 0.0
    ipa = (ips<=config.ip_alvo).sum(); ipd = (ip_novo<=config.ip_alvo).sum()
    rat = reb_atual.sum()

    print("\n"+"="*80)
    print("  SUMARIO FINAL DA OTIMIZACAO")
    print("="*80)
    print(f"  {'Produtos elegiveis:':<40} {len(df):>10,}")
    print(f"  {'Produtos inelegiveis:':<40} {len(df_ine):>10,}")
    print(f"  {'Produtos com rebate > 0:':<40} {nr:>10,}")
    print(f"  {'Produtos com coef < 0 (elasticos):':<40} {(coefs<0).sum():>10,}")
    print("-"*80)
    print(f"  {'GMV ATUAL:':<40} R$ {gmv_s:>14,.2f}")
    print(f"  {'GMV OTIMIZADO:':<40} R$ {gmv_t:>14,.2f}")
    print(f"  {'DELTA GMV (+):':<40} R$ {gmv_t-gmv_s:>14,.2f}")
    if gmv_s>0: print(f"  {'Crescimento GMV:':<40} {(gmv_t-gmv_s)/gmv_s*100:>13.2f}%")
    print("-"*80)
    print(f"  {'Rebate ATUAL (rebate_prazo_magalu):':<40} R$ {rat:>14,.2f}")
    print(f"  {'Rebate OTIMIZADO (novo):':<40} R$ {custo_t:>14,.2f}")
    print(f"  {'Orcamento disponivel:':<40} R$ {config.orcamento_rebate:>14,.2f}")
    print(f"  {'Orcamento utilizado:':<40} {custo_t/config.orcamento_rebate*100:>13.2f}%")
    print("-"*80)
    print(f"  {'Receita Liquida (GMV-Rebate):':<40} R$ {receita_t:>14,.2f}")
    print(f"  {'Desconto Medio (c/ rebate):':<40} {dm:>13.2f}%")
    if custo_t>0: print(f"  {'ROI do Rebate (deltaGMV/custo):':<40} {(gmv_t-gmv_s)/custo_t:>13.2f}x")
    print("-"*80)
    print(f"  {'IP:':<40}")
    print(f"  {'  Competitivos ANTES:':<40} {ipa:>10,}")
    print(f"  {'  Competitivos DEPOIS:':<40} {ipd:>10,}")
    print(f"  {'  IPs corrigidos:':<40} {ipd-ipa:>10,}")
    print(f"  {'  IP medio antes:':<40} {ips.mean():>13.4f}")
    print(f"  {'  IP medio depois:':<40} {ip_novo.mean():>13.4f}")
    print("="*80)
    return resultado

def otimizar_rebate(df_input, orcamento_rebate, max_desconto_pct=0.10, max_mult_demanda=2.0, piso_demanda_min=3.0, ip_alvo=1.0, max_iter=1_000_000, n_restarts=5, seed=42, is_spark=True):
    config = OptConfig(orcamento_rebate=orcamento_rebate, max_desconto_pct=max_desconto_pct, max_mult_demanda=max_mult_demanda, piso_demanda_min=piso_demanda_min, ip_alvo=ip_alvo, max_iter=max_iter, n_restarts=n_restarts, seed=seed)
    df = preparar_dados(df_input, is_spark=is_spark)
    df_el, df_ine = filtrar_elegiveis(df)
    diagnostico(df_el, df_ine)
    if len(df_el)==0: print("\n  ERRO: Nenhum produto elegivel!"); return pd.DataFrame(), []
    rebates, hist = simulated_annealing(df_el, config)
    resultado = gerar_relatorio(df_el, df_ine, rebates, config)
    return resultado, hist

# =====================================================================
# EXECUCAO
# =====================================================================
resultado_ud_mensal, hist_ud_m = otimizar_rebate(
    df_input=df_mensal,
    orcamento_rebate=800_000,
    max_desconto_pct=0.10,
    max_mult_demanda=2.0,
    piso_demanda_min=3.0,
    ip_alvo=0.98,
    max_iter=1_000_000,
    n_restarts=5,
    seed=42,
    is_spark=True,
)

print("\n  TOP 30 PRODUTOS POR GMV OTIMIZADO:")
print("-"*190)
cols = ["navigation_id","sku","categoria_comercial","preco_atual","preco_concorrente","IP_atual","rebate_otimizado","desconto_pct","preco_otimizado","IP_novo","status_ip","qtd_atual","qtd_prevista","mult_demanda","gmv_otimizado","custo_rebate","roi_rebate"]
print(resultado[cols].head(30).to_string(index=False, float_format="%.2f"))

print("\n\n  SUMARIO POR CATEGORIA:")
print("-"*140)
sumario = resultado.groupby("categoria_comercial").agg(
    n_produtos=("navigation_id","count"), rebate_total=("custo_rebate","sum"),
    gmv_atual=("gmv_atual","sum"), gmv_otimizado=("gmv_otimizado","sum"),
    delta_gmv=("delta_gmv","sum"), desc_medio=("desconto_pct","mean"),
    mult_medio=("mult_demanda","mean"), ip_medio_novo=("IP_novo","mean"),
    n_competitivos=("status_ip", lambda x: (x=="COMPETITIVO").sum()),
).round(2)
print(sumario.to_string())

print("\n\n  ANALISE DE IP POR STATUS:")
print("-"*70)
ip_a = resultado.groupby("status_ip").agg(
    n_produtos=("navigation_id","count"), ip_medio=("IP_novo","mean"),
    rebate_medio=("rebate_otimizado","mean"), gmv_total=("gmv_otimizado","sum"),
).round(2)
print(ip_a.to_string())

                                                                                ]]]


  DIAGNOSTICO DOS DADOS
  Total: 36,327 | Elegiveis: 36,327 | Inelegiveis: 0

  Precos: Min R$0.99 | Med R$134.31 | Max R$35,832.99
  Vendas: Min 1 | Med 2 | Max 5703 | Total 353,618
  Coef: Negativo (elastico) 34,717 | Zero (sem efeito) 1,610
  IP: Medio 1.0000 | >1 (nao compet.) 2 | <=1 (compet.) 36,325

  GMV atual: R$ 301,633,448.49
  Rebate atual (rebate_prazo_magalu): R$ 585,783.44

  SIMULATED ANNEALING 
  Produtos: 36,327 (34,717 elasticos, 1,610 sem efeito)
  Orcamento: R$ 800,000.00 | Max desc: 10.0%
  Cap: 2x PROPORCIONAL | Piso: 3 un PROPORCIONAL
  IP alvo: 0.98 | Restarts: 5 | Iter/restart: 200,000

  Solucoes iniciais:
    Greedy IP                 Score:255,182,276.52 | GMV:R$253,869,276.51 | Rebate:R$799,999.99
    Greedy Elasticidade       Score:251,881,530.37 | GMV:R$252,673,030.15 | Rebate:R$799,999.78
    Greedy GMV                Score:251,142,121.49 | GMV:R$251,940,121.49 | Rebate:R$800,000.00
    Aleatoria 1               Score:252,271,487.56 | GMV:R$252,814,346

26/04/14 11:25:33 ERROR TaskSchedulerImpl: Lost executor 7 on 10.251.193.162: 
The executor with id 7 exited with exit code 143(unexpected).
The API gave the following brief reason: Terminated
The API gave the following message: Pod was terminated in response to imminent node shutdown.

The API gave the following container statuses:


	 container name: spark-kubernetes-executor
	 container image: gcr.io/maga-bigdata/dataops/cloud-magalu/spark-jupyter-pyspark-arcade-gcp-3.2.4:1.2.1
	 container state: terminated
	 container started at: 2026-04-14T10:45:52Z
	 container finished at: 2026-04-14T11:25:32Z
	 exit code: 143
	 termination reason: Error
      


    [ 80.0%] Score263,284,399.15 | GMV R$255,531,402.66 | Rebate R$799,999.51 | Temp      0.03
  Restart 1 fim: Score263,800,364.06 | GMV R$255,620,866.90 | Rebate R$799,998.84

  --- Restart 2/5 [Greedy Elasticidade] ---
    [  0.0%] Score252,334,837.00 | GMV R$252,624,201.72 | Rebate R$638,864.72 | Temp  49995.57
    [ 20.0%] Score257,075,762.83 | GMV R$254,246,622.01 | Rebate R$799,859.11 | Temp   1442.57
    [ 40.0%] Score262,327,439.56 | GMV R$255,099,403.76 | Rebate R$799,964.09 | Temp     41.62
    [ 60.0%] Score263,392,236.91 | GMV R$255,279,214.79 | Rebate R$799,977.87 | Temp      1.20
    [ 80.0%] Score263,968,296.34 | GMV R$255,391,794.19 | Rebate R$799,997.36 | Temp      0.03
  Restart 2 fim: Score264,467,046.97 | GMV R$255,487,046.73 | Rebate R$799,999.27

  --- Restart 3/5 [Greedy GMV] ---
    [  0.0%] Score251,628,072.84 | GMV R$252,064,702.73 | Rebate R$649,129.89 | Temp  49995.57


26/04/14 11:55:54 ERROR TaskSchedulerImpl: Lost executor 10 on 10.251.194.163: 
The executor with id 10 exited with exit code 143(unexpected).
The API gave the following brief reason: Terminated
The API gave the following message: Pod was terminated in response to imminent node shutdown.

The API gave the following container statuses:


	 container name: spark-kubernetes-executor
	 container image: gcr.io/maga-bigdata/dataops/cloud-magalu/spark-jupyter-pyspark-arcade-gcp-3.2.4:1.2.1
	 container state: terminated
	 container started at: 2026-04-14T11:26:20Z
	 container finished at: 2026-04-14T11:55:47Z
	 exit code: 143
	 termination reason: Error
      


    [ 20.0%] Score256,942,130.92 | GMV R$254,324,629.82 | Rebate R$799,996.44 | Temp   1442.57
    [ 40.0%] Score262,207,397.17 | GMV R$255,116,861.57 | Rebate R$799,964.23 | Temp     41.62
    [ 60.0%] Score263,236,881.93 | GMV R$255,285,382.66 | Rebate R$799,999.41 | Temp      1.20
    [ 80.0%] Score263,793,713.38 | GMV R$255,388,206.78 | Rebate R$799,992.08 | Temp      0.03
  Restart 3 fim: Score264,266,892.20 | GMV R$255,472,391.65 | Rebate R$799,998.13

  --- Restart 4/5 [Aleatoria 1] ---
    [  0.0%] Score253,423,003.45 | GMV R$253,684,153.16 | Rebate R$727,649.32 | Temp  49995.57
    [ 20.0%] Score256,942,785.13 | GMV R$254,380,391.00 | Rebate R$799,605.60 | Temp   1442.57
    [ 40.0%] Score262,077,447.66 | GMV R$255,128,379.07 | Rebate R$799,930.66 | Temp     41.62
    [ 60.0%] Score263,092,603.15 | GMV R$255,298,100.89 | Rebate R$799,997.06 | Temp      1.20
    [ 80.0%] Score263,704,134.84 | GMV R$255,414,635.46 | Rebate R$799,999.93 | Temp      0.03
  Restart 4 fim: Score264,

NameError: name 'resultado' is not defined

In [45]:
# TOP 30
cols = ["navigation_id","sku","categoria_comercial","preco_atual","preco_concorrente",
        "IP_atual","rebate_otimizado","desconto_pct","preco_otimizado","IP_novo",
        "status_ip","qtd_atual","qtd_prevista","mult_demanda","gmv_otimizado",
        "custo_rebate","roi_rebate"]
print(resultado_ud_mensal[cols].head(30).to_string(index=False, float_format="%.2f"))

# SUMÁRIO POR CATEGORIA
print("\n\n  SUMARIO POR CATEGORIA:")
sumario = resultado_ud_mensal.groupby("categoria_comercial").agg(
    n_produtos=("navigation_id","count"),
    rebate_total=("custo_rebate","sum"),
    gmv_atual=("gmv_atual","sum"),
    gmv_otimizado=("gmv_otimizado","sum"),
    delta_gmv=("delta_gmv","sum"),
    desc_medio=("desconto_pct","mean"),
).round(2)
print(sumario.to_string())

# SALVAR EM EXCEL
resultado_ud_mensal.to_excel("resultado_mensal_final2.xlsx", index=False)
print("Salvo: resultado_mensal_final2.xlsx")

navigation_id             sku categoria_comercial  preco_atual  preco_concorrente  IP_atual  rebate_otimizado  desconto_pct  preco_otimizado  IP_novo status_ip  qtd_atual  qtd_prevista  mult_demanda  gmv_otimizado  custo_rebate  roi_rebate
   efg0410e26            8662                  IN      2149.00            2149.00      1.00              0.00          0.00          2149.00     1.00  MARGINAL    1415.00       1415.00          1.00     3040835.00          0.00        0.00
   jf0kk899g5            8766                  TE      2549.00            2549.00      1.00              0.00          0.00          2549.00     1.00  MARGINAL    1197.00       1112.00          0.93     2834488.00          0.00        0.00
   ebb4eg9b77      83NS0001BR                  IN      4073.03            4073.03      1.00              0.00          0.00          4073.03     1.00  MARGINAL     757.00        682.00          0.90     2777806.46          0.00        0.00
   bj393fc1je         1032877           

26/04/14 13:14:28 ERROR TaskSchedulerImpl: Lost executor 8 on 10.251.192.226: 
The executor with id 8 exited with exit code 143(unexpected).
The API gave the following brief reason: Terminated
The API gave the following message: Pod was terminated in response to imminent node shutdown.

The API gave the following container statuses:


	 container name: spark-kubernetes-executor
	 container image: gcr.io/maga-bigdata/dataops/cloud-magalu/spark-jupyter-pyspark-arcade-gcp-3.2.4:1.2.1
	 container state: terminated
	 container started at: 2026-04-14T10:45:57Z
	 container finished at: 2026-04-14T13:13:59Z
	 exit code: 143
	 termination reason: Error
      
26/04/14 14:06:30 ERROR TaskSchedulerImpl: Lost executor 9 on 10.251.192.162: 
The executor with id 9 exited with exit code 143(unexpected).
The API gave the following brief reason: Terminated
The API gave the following message: Pod was terminated in response to imminent node shutdown.

The API gave the following container statuses:


	 co

In [47]:
@dataclass
class OptConfig:
    orcamento_rebate: float
    max_desconto_pct: float
    max_mult_demanda: float = 2.0
    piso_demanda_min: float = 3.0
    ip_alvo: float = 1.0
    peso_ip: float = 5.0
    temp_inicial: float = 50000.0
    temp_final: float = 0.001
    max_iter: int = 1_000_000
    n_restarts: int = 5
    iter_por_restart: int = 0
    penalidade_orcamento: float = 100.0
    seed: Optional[int] = 42
    def __post_init__(self):
        if self.iter_por_restart == 0:
            self.iter_por_restart = self.max_iter // self.n_restarts
    @property
    def taxa_resfriamento(self) -> float:
        return (self.temp_final / self.temp_inicial) ** (1.0 / max(self.iter_por_restart, 1))

def preparar_dados(df_input, is_spark=True):
    # Mapeamento: nome atual → nome que o otimizador usa internamente
    rename_map = {
        "categoria": "categoria_comercial",
        "preco_por": "preco_por_atual",
        "concorrente": "Concorrente",
        "carteira": "Carteira",
        "rebate_unitario_medio": "rebate_prazo_magalu",
        "seller_id": "SellerID",
    }
    
    cols_obr = ["navigation_id", "categoria_comercial", "preco_por_atual",
                "quantidadeitens", "coef", "intercept", "SellerID", "estoque", "IP"]
    cols_opt = ["Concorrente", "Carteira", "rebate_prazo_magalu"]

    if is_spark:
        # Renomear colunas antes de converter
        for old, new in rename_map.items():
            if old in df_input.columns:
                df_input = df_input.withColumnRenamed(old, new)
        cols_usar = [c for c in cols_obr + cols_opt if c in df_input.columns]
        df = df_input.select(cols_usar).toPandas()
    else:
        df = df_input.rename(columns=rename_map)
        cols_usar = [c for c in cols_obr + cols_opt if c in df.columns]
        df = df[cols_usar].copy()

    df = df.dropna(subset=["coef", "preco_por_atual"])
    df = df[df["preco_por_atual"] > 0].copy()
    df["preco_por_atual"] = df["preco_por_atual"].astype(float)
    df["coef"] = df["coef"].astype(float)
    df["intercept"] = df["intercept"].fillna(0).astype(float)
    df["quantidadeitens"] = df["quantidadeitens"].fillna(0).astype(float)
    df["estoque"] = df["estoque"].fillna(9999).astype(int)
    df["IP"] = df["IP"].fillna(1.0).astype(float).replace(0.0, 1.0)
    df["preco_concorrente"] = np.where(
        df["IP"] > 0, df["preco_por_atual"] / df["IP"], df["preco_por_atual"]
    )
    for col, default in [("Concorrente", "N/A"), ("Carteira", "N/A"), ("rebate_prazo_magalu", 0.0)]:
        if col not in df.columns:
            df[col] = default
        else:
            df[col] = df[col].fillna(default)
    df["rebate_prazo_magalu"] = pd.to_numeric(df["rebate_prazo_magalu"], errors="coerce").fillna(0.0)
    df = df.drop_duplicates(subset=["navigation_id"], keep="first").reset_index(drop=True)
    return df

def filtrar_elegiveis(df):
    mask = (df["quantidadeitens"]>0) & (df["estoque"]>0) & (df["preco_por_atual"]>0)
    return df[mask].reset_index(drop=True), df[~mask].reset_index(drop=True)

def diagnostico(df, df_ine):
    print("\n" + "="*80)
    print("  DIAGNOSTICO DOS DADOS")
    print("="*80)
    print(f"  Total: {len(df)+len(df_ine):,} | Elegiveis: {len(df):,} | Inelegiveis: {len(df_ine):,}")
    if len(df_ine)>0:
        print(f"    Sem vendas: {(df_ine['quantidadeitens']<=0).sum():,} | Sem estoque: {(df_ine['estoque']<=0).sum():,}")
    print(f"\n  Precos: Min R${df['preco_por_atual'].min():,.2f} | Med R${df['preco_por_atual'].median():,.2f} | Max R${df['preco_por_atual'].max():,.2f}")
    print(f"  Vendas: Min {df['quantidadeitens'].min():.0f} | Med {df['quantidadeitens'].median():.0f} | Max {df['quantidadeitens'].max():.0f} | Total {df['quantidadeitens'].sum():,.0f}")
    c = df["coef"]
    print(f"  Coef: Negativo (elastico) {(c<0).sum():,} | Zero (sem efeito) {(c==0).sum():,}")
    print(f"  IP: Medio {df['IP'].mean():.4f} | >1 (nao compet.) {(df['IP']>1).sum():,} | <=1 (compet.) {(df['IP']<=1).sum():,}")
    print(f"\n  GMV atual: R$ {(df['preco_por_atual']*df['quantidadeitens']).sum():,.2f}")
    print(f"  Rebate atual (rebate_prazo_magalu): R$ {df['rebate_prazo_magalu'].sum():,.2f}")
    print("="*80)

# =====================================================================
# MODELO DE DEMANDA - CAP PROPORCIONAL
# =====================================================================
def prever_demanda(precos, rebates, qtds, coefs, estoques, max_mult=2.0, piso_min=3.0, max_desc_pct=0.10):
    """
    Q_nova = qtd_atual - coef * rebate  (coef negativo -> desconto aumenta demanda)

    Cap PROPORCIONAL ao desconto:
      fator = desconto_dado / desconto_maximo  (0 a 1)
      mult_max = 1 + fator * (max_mult - 1)
      piso = qtd + piso_min * fator

    Desc 10% -> mult 2.0x | Desc 5% -> mult 1.5x | Desc 1% -> mult 1.1x | Desc 0.2% -> mult 1.02x
    """
    r = np.maximum(rebates, 0.0)
    qtd_nova = qtds + coefs * (-r)
    desc_pct = r / np.maximum(precos, 0.01)
    fator = np.clip(desc_pct / max(max_desc_pct, 0.001), 0.0, 1.0)
    mult = 1.0 + fator * (max_mult - 1.0)
    piso = qtds + piso_min * fator
    max_qtd = np.maximum(qtds * mult, piso)
    teto = np.minimum(max_qtd, estoques.astype(float))
    return np.clip(qtd_nova, 0.0, teto)

def calc_metricas(precos, rebates, qtds, coefs, estoques, mm=2.0, pm=3.0, mdp=0.10):
    r = np.maximum(rebates, 0.0)
    po = np.maximum(precos - r, 0.01)
    qn = prever_demanda(precos, r, qtds, coefs, estoques, mm, pm, mdp)
    gmv = (po * qn).sum()
    custo = (r * qn).sum()
    return po, qn, gmv, max(custo, 0.0), gmv - max(custo, 0.0)

def func_obj(precos, rebates, qtds, coefs, estoques, pc, ips, cfg):
    r = np.maximum(rebates, 0.0)
    _, _, gmv, custo, receita = calc_metricas(precos, r, qtds, coefs, estoques, cfg.max_mult_demanda, cfg.piso_demanda_min, cfg.max_desconto_pct)
    score = receita
    if custo > cfg.orcamento_rebate:
        exc = custo - cfg.orcamento_rebate
        score -= cfg.penalidade_orcamento * exc + 0.1 * exc**2
    viol = np.maximum(r/np.maximum(precos,0.01) - cfg.max_desconto_pct, 0.0)
    if viol.sum() > 0: score -= 5000.0 * viol.sum()
    po = np.maximum(precos - r, 0.01)
    ip_novo = po / np.maximum(pc, 0.01)
    score += cfg.peso_ip * ((ips > cfg.ip_alvo) & (ip_novo <= cfg.ip_alvo)).sum() * 100
    score -= 0.3 * r[ips <= cfg.ip_alvo * 0.85].sum()
    return score

# =====================================================================
# SOLUCOES INICIAIS
# =====================================================================
def _ok(coef): return coef < 0

def sol_greedy_ip(precos, qtds, coefs, est, ips, pc, mdp, orc, ip_alvo, mm, pm):
    n = len(precos); mr = precos*mdp; reb = np.zeros(n); rest = orc
    for i in np.argsort(-ips):
        if rest <= 0: break
        if not _ok(coefs[i]) or ips[i] <= ip_alvo*0.9: continue
        ri = min(max(0.0, precos[i]-ip_alvo*pc[i]), mr[i])
        q = prever_demanda(precos[i:i+1], np.array([ri]), qtds[i:i+1], coefs[i:i+1], est[i:i+1], mm, pm, mdp)[0]
        c = ri*q
        if 0 < c <= rest: reb[i]=ri; rest-=c
    return reb

def sol_greedy_elast(precos, qtds, coefs, est, mdp, orc, mm, pm):
    n = len(precos); mr = precos*mdp; reb = np.zeros(n); rest = orc
    elast = np.abs(np.minimum(coefs, 0))
    for i in np.argsort(-elast):
        if rest <= 0 or not _ok(coefs[i]): break
        ri = mr[i]
        q = prever_demanda(precos[i:i+1], np.array([ri]), qtds[i:i+1], coefs[i:i+1], est[i:i+1], mm, pm, mdp)[0]
        c = ri*q
        if 0 < c <= rest: reb[i]=ri; rest-=c
    return reb

def sol_greedy_gmv(precos, qtds, coefs, est, mdp, orc, mm, pm):
    n = len(precos); mr = precos*mdp; reb = np.zeros(n); rest = orc
    gp = np.zeros(n)
    for i in range(n):
        if not _ok(coefs[i]): continue
        q = prever_demanda(precos[i:i+1], mr[i:i+1], qtds[i:i+1], coefs[i:i+1], est[i:i+1], mm, pm, mdp)[0]
        gp[i] = (precos[i]-mr[i])*q
    for i in np.argsort(-gp):
        if rest <= 0 or not _ok(coefs[i]): break
        q = prever_demanda(precos[i:i+1], np.array([mr[i]]), qtds[i:i+1], coefs[i:i+1], est[i:i+1], mm, pm, mdp)[0]
        c = mr[i]*q
        if 0 < c <= rest: reb[i]=mr[i]; rest-=c
    return reb

def sol_rand(precos, coefs, mdp):
    mr = precos*mdp; mask = coefs<0; r = np.zeros(len(precos))
    r[mask] = np.random.uniform(0,1,mask.sum())*mr[mask]
    return r

def projetar(reb, precos, qtds, coefs, est, orc, mdp, mm, pm):
    reb = np.clip(reb, 0, precos*mdp); reb[coefs>=0]=0.0
    _, _, _, custo, _ = calc_metricas(precos, reb, qtds, coefs, est, mm, pm, mdp)
    if custo > orc and custo > 0:
        reb *= orc/custo*0.95; reb = np.clip(reb, 0, precos*mdp); reb[coefs>=0]=0.0
    return reb

# =====================================================================
# SIMULATED ANNEALING
# =====================================================================
def gerar_vizinho(reb, precos, qtds, coefs, est, pc, ips, mdp, orc, ip_alvo, temp, t0, mm, pm):
    novo = reb.copy(); n = len(reb); fator = temp/t0; mr = precos*mdp
    np_ = max(1, int(min(n, max(3, n*0.1))*fator+1))
    for i in random.sample(range(n), min(np_, n)):
        if mr[i]<=0 or coefs[i]>=0: novo[i]=0.0; continue
        r = random.random()
        if r<0.05: novo[i]=random.uniform(0,mr[i])
        elif r<0.10: novo[i]=0.0
        elif r<0.15: novo[i]=mr[i]
        elif r<0.25:
            rip = min(max(0.0, precos[i]-ip_alvo*pc[i]), mr[i])
            novo[i]=np.clip(rip+random.gauss(0,max(mr[i]*fator*0.15,0.001)),0,mr[i])
        elif r<0.35:
            j = random.randint(0,n-1)
            if j!=i and coefs[j]<0:
                t=novo[i]+novo[j]; s=random.uniform(0,1)
                novo[i]=min(s*t,mr[i]); novo[j]=min((1-s)*t,mr[j])
        else:
            sig = mr[i]*fator*0.3+mr[i]*0.02
            novo[i]=np.clip(novo[i]+random.gauss(0,max(sig,0.001)),0,mr[i])
    novo = np.maximum(novo, 0.0)
    return projetar(novo, precos, qtds, coefs, est, orc, mdp, mm, pm)

def simulated_annealing(df, config):
    if config.seed is not None: random.seed(config.seed); np.random.seed(config.seed)
    precos=df["preco_por_atual"].values; qtds=df["quantidadeitens"].values
    coefs=df["coef"].values; est=df["estoque"].values
    ips=df["IP"].values; pc=df["preco_concorrente"].values
    taxa=config.taxa_resfriamento; mm=config.max_mult_demanda; pm=config.piso_demanda_min; mdp=config.max_desconto_pct; n=len(df)

    sols = [
        ("Greedy IP", sol_greedy_ip(precos,qtds,coefs,est,ips,pc,mdp,config.orcamento_rebate,config.ip_alvo,mm,pm)),
        ("Greedy Elasticidade", sol_greedy_elast(precos,qtds,coefs,est,mdp,config.orcamento_rebate,mm,pm)),
        ("Greedy GMV", sol_greedy_gmv(precos,qtds,coefs,est,mdp,config.orcamento_rebate,mm,pm)),
        ("Aleatoria 1", sol_rand(precos,coefs,mdp)),
        ("Aleatoria 2", sol_rand(precos,coefs,mdp)),
    ]
    for i,(nm,s) in enumerate(sols):
        sols[i] = (nm, projetar(np.maximum(s,0), precos, qtds, coefs, est, config.orcamento_rebate, mdp, mm, pm))

    print("\n"+"="*80)
    print("  SIMULATED ANNEALING v7 - Cap Proporcional ao Desconto")
    print("="*80)
    print(f"  Produtos: {n:,} ({(coefs<0).sum():,} elasticos, {(coefs>=0).sum():,} sem efeito)")
    print(f"  Orcamento: R$ {config.orcamento_rebate:,.2f} | Max desc: {mdp*100:.1f}%")
    print(f"  Cap: {mm:.0f}x PROPORCIONAL | Piso: {pm:.0f} un PROPORCIONAL")
    print(f"  IP alvo: {config.ip_alvo:.2f} | Restarts: {config.n_restarts} | Iter/restart: {config.iter_por_restart:,}")
    print("="*80)

    print(f"\n  Solucoes iniciais:")
    for nm,s in sols:
        sc=func_obj(precos,s,qtds,coefs,est,pc,ips,config)
        _,_,g,c,_=calc_metricas(precos,s,qtds,coefs,est,mm,pm,mdp)
        print(f"    {nm:<25} Score:{sc:>14,.2f} | GMV:R${g:>14,.2f} | Rebate:R${c:>10,.2f}")

    best_reb=None; best_sc=-float("inf"); hist=[]; it_total=0

    for ri in range(config.n_restarts):
        nm,si = sols[ri % len(sols)]
        cur=si.copy(); cur_sc=func_obj(precos,cur,qtds,coefs,est,pc,ips,config)
        br=cur.copy(); br_sc=cur_sc; temp=config.temp_inicial; stag=0
        print(f"\n  --- Restart {ri+1}/{config.n_restarts} [{nm}] ---")
        li = max(config.iter_por_restart//5, 1)

        for it in range(config.iter_por_restart):
            nv = gerar_vizinho(cur,precos,qtds,coefs,est,pc,ips,mdp,config.orcamento_rebate,config.ip_alvo,temp,config.temp_inicial,mm,pm)
            nv_sc = func_obj(precos,nv,qtds,coefs,est,pc,ips,config)
            d = nv_sc - cur_sc
            if d>0 or random.random()<math.exp(d/max(temp,1e-10)):
                cur=nv; cur_sc=nv_sc
                if cur_sc>br_sc: br_sc=cur_sc; br=cur.copy(); stag=0
                else: stag+=1
            else: stag+=1
            temp*=taxa; it_total+=1

            if it%li==0:
                _,_,g,c,r=calc_metricas(precos,br,qtds,coefs,est,mm,pm,mdp)
                pct=it/config.iter_por_restart*100
                hist.append({"restart":ri,"iter":it_total,"temp":temp,"score":br_sc,"gmv":g,"rebate":c})
                print(f"    [{pct:5.1f}%] Score{br_sc:>14,.2f} | GMV R${g:>14,.2f} | Rebate R${c:>10,.2f} | Temp{temp:>10.2f}")

            if stag > config.iter_por_restart//10:
                sr=sol_rand(precos,coefs,mdp)
                sr=projetar(np.maximum(sr,0),precos,qtds,coefs,est,config.orcamento_rebate,mdp,mm,pm)
                cur=0.7*br+0.3*sr; cur=np.maximum(cur,0)
                cur=projetar(cur,precos,qtds,coefs,est,config.orcamento_rebate,mdp,mm,pm)
                cur_sc=func_obj(precos,cur,qtds,coefs,est,pc,ips,config); stag=0

        if br_sc>best_sc: best_sc=br_sc; best_reb=br.copy()
        _,_,g,c,_=calc_metricas(precos,br,qtds,coefs,est,mm,pm,mdp)
        print(f"  Restart {ri+1} fim: Score{br_sc:>14,.2f} | GMV R${g:>14,.2f} | Rebate R${c:>10,.2f}")

    print("\n"+"="*80)
    _,_,gf,cf,rf=calc_metricas(precos,best_reb,qtds,coefs,est,mm,pm,mdp)
    print(f"  MELHOR: Score{best_sc:>14,.2f} | GMV:R${gf:>14,.2f} | Rebate:R${cf:>10,.2f} | Receita:R${rf:>14,.2f}")
    print("="*80)
    return best_reb, hist

# =====================================================================
# RELATORIO
# =====================================================================
def gerar_relatorio(df, df_ine, rebates, config):
    precos=df["preco_por_atual"].values; qtds=df["quantidadeitens"].values
    coefs=df["coef"].values; est=df["estoque"].values
    ips=df["IP"].values; pc=df["preco_concorrente"].values; reb_atual=df["rebate_prazo_magalu"].values
    rebates=np.maximum(rebates,0.0); mm=config.max_mult_demanda; pm=config.piso_demanda_min; mdp=config.max_desconto_pct

    po,qc,gmv_t,custo_t,receita_t = calc_metricas(precos,rebates,qtds,coefs,est,mm,pm,mdp)
    _,qs,gmv_s,_,_ = calc_metricas(precos,np.zeros(len(df)),qtds,coefs,est,mm,pm,mdp)

    ip_novo = po/np.maximum(pc,0.01)
    custo_pp = np.maximum(rebates*qc, 0.0)
    gmv_pp_com = po*qc; gmv_pp_sem = precos*qs

    resultado = pd.DataFrame({
        "navigation_id": df["navigation_id"].values,
        "categoria_comercial": df["categoria_comercial"].values,
        "seller_id": df["SellerID"].values,
        "concorrente": df["Concorrente"].values if "Concorrente" in df.columns else "N/A",
        "carteira": df["Carteira"].values if "Carteira" in df.columns else "N/A",
        "preco_atual": precos.round(2),
        "preco_concorrente": pc.round(2),
        "IP_atual": ips.round(4),
        "rebate_atual_magalu": reb_atual.round(2),
        "rebate_otimizado": rebates.round(2),
        "desconto_pct": ((rebates/np.maximum(precos,0.01))*100).round(2),
        "preco_otimizado": po.round(2),
        "IP_novo": ip_novo.round(4),
        "delta_IP": (ip_novo - ips).round(4),
        "status_ip": np.where(ip_novo<=config.ip_alvo,"COMPETITIVO",np.where(ip_novo<=config.ip_alvo*1.05,"MARGINAL","NAO_COMPETITIVO")),
        "qtd_atual": qtds.round(0),
        "qtd_prevista": qc.round(1),
        "delta_qtd": (qc-qtds).round(1),
        "mult_demanda": np.where(qtds>0, (qc/qtds).round(2), 0.0),
        "gmv_atual": gmv_pp_sem.round(2),
        "gmv_otimizado": gmv_pp_com.round(2),
        "delta_gmv": (gmv_pp_com-gmv_pp_sem).round(2),
        "custo_rebate": custo_pp.round(2),
        "lucro_previsto": (gmv_pp_com-custo_pp).round(2),
        "roi_rebate": np.where(custo_pp>0.01, np.round((gmv_pp_com-gmv_pp_sem)/np.maximum(custo_pp,0.01),2), 0.0),
        "estoque": est,
    })
    resultado = resultado.sort_values("gmv_otimizado", ascending=False).reset_index(drop=True)

    nr = (rebates>0.01).sum()
    mk = resultado["rebate_otimizado"]>0.01
    dm = resultado.loc[mk,"desconto_pct"].mean() if mk.any() else 0.0
    ipa = (ips<=config.ip_alvo).sum(); ipd = (ip_novo<=config.ip_alvo).sum()
    rat = reb_atual.sum()

    print("\n"+"="*80)
    print("  SUMARIO FINAL DA OTIMIZACAO")
    print("="*80)
    print(f"  {'Produtos elegiveis:':<40} {len(df):>10,}")
    print(f"  {'Produtos inelegiveis:':<40} {len(df_ine):>10,}")
    print(f"  {'Produtos com rebate > 0:':<40} {nr:>10,}")
    print(f"  {'Produtos com coef < 0 (elasticos):':<40} {(coefs<0).sum():>10,}")
    print("-"*80)
    print(f"  {'GMV ATUAL:':<40} R$ {gmv_s:>14,.2f}")
    print(f"  {'GMV OTIMIZADO:':<40} R$ {gmv_t:>14,.2f}")
    print(f"  {'DELTA GMV (+):':<40} R$ {gmv_t-gmv_s:>14,.2f}")
    if gmv_s>0: print(f"  {'Crescimento GMV:':<40} {(gmv_t-gmv_s)/gmv_s*100:>13.2f}%")
    print("-"*80)
    print(f"  {'Rebate ATUAL (rebate_prazo_magalu):':<40} R$ {rat:>14,.2f}")
    print(f"  {'Rebate OTIMIZADO (novo):':<40} R$ {custo_t:>14,.2f}")
    print(f"  {'Orcamento disponivel:':<40} R$ {config.orcamento_rebate:>14,.2f}")
    print(f"  {'Orcamento utilizado:':<40} {custo_t/config.orcamento_rebate*100:>13.2f}%")
    print("-"*80)
    print(f"  {'Receita Liquida (GMV-Rebate):':<40} R$ {receita_t:>14,.2f}")
    print(f"  {'Desconto Medio (c/ rebate):':<40} {dm:>13.2f}%")
    if custo_t>0: print(f"  {'ROI do Rebate (deltaGMV/custo):':<40} {(gmv_t-gmv_s)/custo_t:>13.2f}x")
    print("-"*80)
    print(f"  {'IP:':<40}")
    print(f"  {'  Competitivos ANTES:':<40} {ipa:>10,}")
    print(f"  {'  Competitivos DEPOIS:':<40} {ipd:>10,}")
    print(f"  {'  IPs corrigidos:':<40} {ipd-ipa:>10,}")
    print(f"  {'  IP medio antes:':<40} {ips.mean():>13.4f}")
    print(f"  {'  IP medio depois:':<40} {ip_novo.mean():>13.4f}")
    print("="*80)
    return resultado

def otimizar_rebate(df_input, orcamento_rebate, max_desconto_pct=0.10, max_mult_demanda=2.0, piso_demanda_min=3.0, ip_alvo=1.0, max_iter=1_000_000, n_restarts=5, seed=42, is_spark=True):
    config = OptConfig(orcamento_rebate=orcamento_rebate, max_desconto_pct=max_desconto_pct, max_mult_demanda=max_mult_demanda, piso_demanda_min=piso_demanda_min, ip_alvo=ip_alvo, max_iter=max_iter, n_restarts=n_restarts, seed=seed)
    df = preparar_dados(df_input, is_spark=is_spark)
    df_el, df_ine = filtrar_elegiveis(df)
    diagnostico(df_el, df_ine)
    if len(df_el)==0: print("\n  ERRO: Nenhum produto elegivel!"); return pd.DataFrame(), []
    rebates, hist = simulated_annealing(df_el, config)
    resultado = gerar_relatorio(df_el, df_ine, rebates, config)
    return resultado, hist

# =====================================================================
# EXECUCAO
# =====================================================================
resultado_ep_mensal, hist_ep_m = otimizar_rebate(
    df_input=df_ep_mensal,
    orcamento_rebate=150_000,
    max_desconto_pct=0.10,
    max_mult_demanda=2.0,
    piso_demanda_min=3.0,
    ip_alvo=0.98,
    max_iter=1_000_000,
    n_restarts=5,
    seed=42,
    is_spark=True,
)

print("\n  TOP 30 PRODUTOS POR GMV OTIMIZADO:")
print("-"*190)
cols = ["navigation_id","categoria_comercial","preco_atual","preco_concorrente",
        "IP_atual","rebate_otimizado","desconto_pct","preco_otimizado","IP_novo",
        "status_ip","qtd_atual","qtd_prevista","mult_demanda","gmv_otimizado",
        "custo_rebate","roi_rebate"]
print(resultado_ep_mensal[cols].head(30).to_string(index=False, float_format="%.2f"))

print("\n\n  SUMARIO POR CATEGORIA:")
print("-"*140)
sumario = resultado_ep_mensal.groupby("categoria_comercial").agg(
    n_produtos=("navigation_id","count"),
    rebate_total=("custo_rebate","sum"),
    gmv_atual=("gmv_atual","sum"),
    gmv_otimizado=("gmv_otimizado","sum"),
    delta_gmv=("delta_gmv","sum"),
    desc_medio=("desconto_pct","mean"),
    mult_medio=("mult_demanda","mean"),
    ip_medio_novo=("IP_novo","mean"),
    n_competitivos=("status_ip", lambda x: (x=="COMPETITIVO").sum()),
).round(2)
print(sumario.to_string())

print("\n\n  ANALISE DE IP POR STATUS:")
print("-"*70)
ip_a = resultado_ep_mensal.groupby("status_ip").agg(
    n_produtos=("navigation_id","count"),
    ip_medio=("IP_novo","mean"),
    rebate_medio=("rebate_otimizado","mean"),
    gmv_total=("gmv_otimizado","sum"),
).round(2)
print(ip_a.to_string())

# Salvar
resultado_ep_mensal.to_csv("resultado_ep_mensal_final.csv", index=False)
print("\nSalvo: resultado_ep_mensal_final.csv")

                                                                                0]]]]


  DIAGNOSTICO DOS DADOS
  Total: 1,379 | Elegiveis: 1,379 | Inelegiveis: 0

  Precos: Min R$9.90 | Med R$159.92 | Max R$2,309.25
  Vendas: Min 1 | Med 2 | Max 65 | Total 4,957
  Coef: Negativo (elastico) 1,365 | Zero (sem efeito) 14
  IP: Medio 1.0233 | >1 (nao compet.) 147 | <=1 (compet.) 1,232

  GMV atual: R$ 1,053,878.99
  Rebate atual (rebate_prazo_magalu): R$ 7,869.22

  SIMULATED ANNEALING v7 - Cap Proporcional ao Desconto
  Produtos: 1,379 (1,365 elasticos, 14 sem efeito)
  Orcamento: R$ 150,000.00 | Max desc: 10.0%
  Cap: 2x PROPORCIONAL | Piso: 3 un PROPORCIONAL
  IP alvo: 0.98 | Restarts: 5 | Iter/restart: 200,000

  Solucoes iniciais:
    Greedy IP                 Score:  1,793,713.07 | GMV:R$  1,319,459.51 | Rebate:R$ 47,746.44
    Greedy Elasticidade       Score:  1,982,380.34 | GMV:R$  1,677,934.31 | Rebate:R$149,999.75
    Greedy GMV                Score:  1,652,363.91 | GMV:R$  1,632,874.49 | Rebate:R$149,999.87
    Aleatoria 1               Score:  1,896,496.68 | GMV

[Stage 444:===================================================(401 + -31) / 400]

    [ 20.0%] Score  1,839,860.67 | GMV R$  1,510,838.03 | Rebate R$ 94,955.21 | Temp   1442.57
    [ 40.0%] Score  2,231,402.40 | GMV R$  1,774,434.62 | Rebate R$149,995.21 | Temp     41.62
    [ 60.0%] Score  2,257,512.12 | GMV R$  1,797,556.04 | Rebate R$149,998.05 | Temp      1.20


[Stage 444:===================================================(401 + -31) / 400]

...

[Stage 444:===================================================(401 + -31) / 400]

  Restart 4 fim: Score  2,266,007.66 | GMV R$  1,807,561.54 | Rebate R$149,999.97

  --- Restart 5/5 [Aleatoria 2] ---
    [  0.0%] Score  1,902,667.59 | GMV R$  1,508,638.93 | Rebate R$ 87,432.71 | Temp  49995.57
    [ 20.0%] Score  1,963,399.40 | GMV R$  1,511,408.28 | Rebate R$ 82,469.88 | Temp   1442.57
    [ 40.0%] Score  2,232,970.21 | GMV R$  1,777,517.17 | Rebate R$149,990.55 | Temp     41.62


[Stage 444:===================================================(401 + -31) / 400]

    [ 60.0%] Score  2,258,716.33 | GMV R$  1,800,776.64 | Rebate R$149,999.79 | Temp      1.20
    [ 80.0%] Score  2,264,490.84 | GMV R$  1,806,048.43 | Rebate R$149,999.98 | Temp      0.03
  Restart 5 fim: Score  2,267,241.04 | GMV R$  1,808,798.37 | Rebate R$149,999.96

  MELHOR: Score  2,267,241.04 | GMV:R$  1,808,798.37 | Rebate:R$149,999.96 | Receita:R$  1,658,798.41

  SUMARIO FINAL DA OTIMIZACAO
  Produtos elegiveis:                           1,379
  Produtos inelegiveis:                             0
  Produtos com rebate > 0:                      1,357
  Produtos com coef < 0 (elasticos):            1,365
--------------------------------------------------------------------------------
  GMV ATUAL:                               R$   1,032,863.02
  GMV OTIMIZADO:                           R$   1,808,798.37
  DELTA GMV (+):                           R$     775,935.35
  Crescimento GMV:                                 75.12%
--------------------------------------------------------

[Stage 444:===================================================(401 + -31) / 400]

In [48]:
import pandas as pd

# Lê os dois arquivos
df_resultado = pd.read_excel("resultado_ud_mensal_final.xlsx", dtype={"navigation_id": str})
df_aceitos = pd.read_excel("skus_aceitos_ud.xlsx", dtype={"SKU": str})

skus_resultado = set(df_resultado["navigation_id"].tolist())
skus_aceitos = set(df_aceitos["SKU"].tolist())

# SKUs que estão no aceitos mas não apareceram no resultado
faltando = skus_aceitos - skus_resultado

# SKUs que estão no resultado mas não estavam na lista aceita
sobrando = skus_resultado - skus_aceitos

print(f"Total aceitos:       {len(skus_aceitos)}")
print(f"Total no resultado:  {len(skus_resultado)}")
print(f"Faltando no resultado: {len(faltando)}")
print(f"Sobrando no resultado: {len(sobrando)}")

Total aceitos:       5795
Total no resultado:  5080
Faltando no resultado: 715
Sobrando no resultado: 0


In [49]:
import pandas as pd

# Lê os dois arquivos
df_resultado = pd.read_excel("resultado_ep_mensal_final.xlsx", dtype={"navigation_id": str})
df_aceitos = pd.read_excel("skus_aceitos_ep.xlsx", dtype={"SKU": str})

skus_resultado = set(df_resultado["navigation_id"].tolist())
skus_aceitos = set(df_aceitos["SKU"].tolist())

# SKUs que estão no aceitos mas não apareceram no resultado
faltando = skus_aceitos - skus_resultado

# SKUs que estão no resultado mas não estavam na lista aceita
sobrando = skus_resultado - skus_aceitos

print(f"Total aceitos:       {len(skus_aceitos)}")
print(f"Total no resultado:  {len(skus_resultado)}")
print(f"Faltando no resultado: {len(faltando)}")
print(f"Sobrando no resultado: {len(sobrando)}")

FileNotFoundError: [Errno 2] No such file or directory: 'resultado_ep_mensal_final.xlsx'

[Stage 444:===================================================(401 + -31) / 400]

In [90]:
@dataclass
class OptConfig:
    orcamento_rebate: float
    max_desconto_pct: float
    max_mult_demanda: float = 2.0
    piso_demanda_min: float = 3.0
    ip_alvo: float = 1.0
    peso_ip: float = 5.0
    temp_inicial: float = 50000.0
    temp_final: float = 0.001
    max_iter: int = 1_000_000
    n_restarts: int = 5
    iter_por_restart: int = 0
    penalidade_orcamento: float = 100.0
    seed: Optional[int] = 42
    def __post_init__(self):
        if self.iter_por_restart == 0:
            self.iter_por_restart = self.max_iter // self.n_restarts
    @property
    def taxa_resfriamento(self) -> float:
        return (self.temp_final / self.temp_inicial) ** (1.0 / max(self.iter_por_restart, 1))

def preparar_dados(df_input, is_spark=True):
    # Mapeamento: nome atual → nome que o otimizador usa internamente
    rename_map = {
        "categoria": "categoria_comercial",
        "preco_por": "preco_por_atual",
        "concorrente": "Concorrente",
        "carteira": "Carteira",
        "rebate_unitario_medio": "rebate_prazo_magalu",
        "seller_id": "SellerID",
    }
    
    cols_obr = ["navigation_id", "categoria_comercial", "preco_por_atual",
                "quantidadeitens", "coef", "intercept", "SellerID", "estoque", "IP"]
    cols_opt = ["Concorrente", "Carteira", "rebate_prazo_magalu"]

    if is_spark:
        # Renomear colunas antes de converter
        for old, new in rename_map.items():
            if old in df_input.columns:
                df_input = df_input.withColumnRenamed(old, new)
        cols_usar = [c for c in cols_obr + cols_opt if c in df_input.columns]
        df = df_input.select(cols_usar).toPandas()
    else:
        df = df_input.rename(columns=rename_map)
        cols_usar = [c for c in cols_obr + cols_opt if c in df.columns]
        df = df[cols_usar].copy()

    df = df.dropna(subset=["coef", "preco_por_atual"])
    df = df[df["preco_por_atual"] > 0].copy()
    df["preco_por_atual"] = df["preco_por_atual"].astype(float)
    df["coef"] = df["coef"].astype(float)
    df["intercept"] = df["intercept"].fillna(0).astype(float)
    df["quantidadeitens"] = df["quantidadeitens"].fillna(0).astype(float)
    df["estoque"] = df["estoque"].fillna(9999).astype(int)
    df["IP"] = df["IP"].fillna(1.0).astype(float).replace(0.0, 1.0)
    df["preco_concorrente"] = np.where(
        df["IP"] > 0, df["preco_por_atual"] / df["IP"], df["preco_por_atual"]
    )
    for col, default in [("Concorrente", "N/A"), ("Carteira", "N/A"), ("rebate_prazo_magalu", 0.0)]:
        if col not in df.columns:
            df[col] = default
        else:
            df[col] = df[col].fillna(default)
    df["rebate_prazo_magalu"] = pd.to_numeric(df["rebate_prazo_magalu"], errors="coerce").fillna(0.0)
    df = df.drop_duplicates(subset=["navigation_id"], keep="first").reset_index(drop=True)
    return df

def filtrar_elegiveis(df):
    mask = (df["quantidadeitens"]>0) & (df["estoque"]>0) & (df["preco_por_atual"]>0)
    return df[mask].reset_index(drop=True), df[~mask].reset_index(drop=True)

def diagnostico(df, df_ine):
    print("\n" + "="*80)
    print("  DIAGNOSTICO DOS DADOS")
    print("="*80)
    print(f"  Total: {len(df)+len(df_ine):,} | Elegiveis: {len(df):,} | Inelegiveis: {len(df_ine):,}")
    if len(df_ine)>0:
        print(f"    Sem vendas: {(df_ine['quantidadeitens']<=0).sum():,} | Sem estoque: {(df_ine['estoque']<=0).sum():,}")
    print(f"\n  Precos: Min R${df['preco_por_atual'].min():,.2f} | Med R${df['preco_por_atual'].median():,.2f} | Max R${df['preco_por_atual'].max():,.2f}")
    print(f"  Vendas: Min {df['quantidadeitens'].min():.0f} | Med {df['quantidadeitens'].median():.0f} | Max {df['quantidadeitens'].max():.0f} | Total {df['quantidadeitens'].sum():,.0f}")
    c = df["coef"]
    print(f"  Coef: Negativo (elastico) {(c<0).sum():,} | Zero (sem efeito) {(c==0).sum():,}")
    print(f"  IP: Medio {df['IP'].mean():.4f} | >1 (nao compet.) {(df['IP']>1).sum():,} | <=1 (compet.) {(df['IP']<=1).sum():,}")
    print(f"\n  GMV atual: R$ {(df['preco_por_atual']*df['quantidadeitens']).sum():,.2f}")
    print(f"  Rebate atual (rebate_prazo_magalu): R$ {df['rebate_prazo_magalu'].sum():,.2f}")
    print("="*80)

# =====================================================================
# MODELO DE DEMANDA - CAP PROPORCIONAL
# =====================================================================
def prever_demanda(precos, rebates, qtds, coefs, estoques, max_mult=2.0, piso_min=3.0, max_desc_pct=0.10):
    """
    Q_nova = qtd_atual - coef * rebate  (coef negativo -> desconto aumenta demanda)

    Cap PROPORCIONAL ao desconto:
      fator = desconto_dado / desconto_maximo  (0 a 1)
      mult_max = 1 + fator * (max_mult - 1)
      piso = qtd + piso_min * fator

    Desc 10% -> mult 2.0x | Desc 5% -> mult 1.5x | Desc 1% -> mult 1.1x | Desc 0.2% -> mult 1.02x
    """
    r = np.maximum(rebates, 0.0)
    qtd_nova = qtds + coefs * (-r)
    desc_pct = r / np.maximum(precos, 0.01)
    fator = np.clip(desc_pct / max(max_desc_pct, 0.001), 0.0, 1.0)
    mult = 1.0 + fator * (max_mult - 1.0)
    piso = qtds + piso_min * fator
    max_qtd = np.maximum(qtds * mult, piso)
    teto = np.minimum(max_qtd, estoques.astype(float))
    return np.clip(qtd_nova, 0.0, teto)

def calc_metricas(precos, rebates, qtds, coefs, estoques, mm=2.0, pm=3.0, mdp=0.10):
    r = np.maximum(rebates, 0.0)
    po = np.maximum(precos - r, 0.01)
    qn = prever_demanda(precos, r, qtds, coefs, estoques, mm, pm, mdp)
    gmv = (po * qn).sum()
    custo = (r * qn).sum()
    return po, qn, gmv, max(custo, 0.0), gmv - max(custo, 0.0)

def func_obj(precos, rebates, qtds, coefs, estoques, pc, ips, cfg):
    r = np.maximum(rebates, 0.0)
    _, _, gmv, custo, receita = calc_metricas(precos, r, qtds, coefs, estoques, cfg.max_mult_demanda, cfg.piso_demanda_min, cfg.max_desconto_pct)
    score = receita
    if custo > cfg.orcamento_rebate:
        exc = custo - cfg.orcamento_rebate
        score -= cfg.penalidade_orcamento * exc + 0.1 * exc**2
    viol = np.maximum(r/np.maximum(precos,0.01) - cfg.max_desconto_pct, 0.0)
    if viol.sum() > 0: score -= 5000.0 * viol.sum()
    po = np.maximum(precos - r, 0.01)
    ip_novo = po / np.maximum(pc, 0.01)
    score += cfg.peso_ip * ((ips > cfg.ip_alvo) & (ip_novo <= cfg.ip_alvo)).sum() * 100
    score -= 0.3 * r[ips <= cfg.ip_alvo * 0.85].sum()
    return score

# =====================================================================
# SOLUCOES INICIAIS
# =====================================================================
def _ok(coef): return coef < 0

def sol_greedy_ip(precos, qtds, coefs, est, ips, pc, mdp, orc, ip_alvo, mm, pm):
    n = len(precos); mr = precos*mdp; reb = np.zeros(n); rest = orc
    for i in np.argsort(-ips):
        if rest <= 0: break
        if not _ok(coefs[i]) or ips[i] <= ip_alvo*0.9: continue
        ri = min(max(0.0, precos[i]-ip_alvo*pc[i]), mr[i])
        q = prever_demanda(precos[i:i+1], np.array([ri]), qtds[i:i+1], coefs[i:i+1], est[i:i+1], mm, pm, mdp)[0]
        c = ri*q
        if 0 < c <= rest: reb[i]=ri; rest-=c
    return reb

def sol_greedy_elast(precos, qtds, coefs, est, mdp, orc, mm, pm):
    n = len(precos); mr = precos*mdp; reb = np.zeros(n); rest = orc
    elast = np.abs(np.minimum(coefs, 0))
    for i in np.argsort(-elast):
        if rest <= 0 or not _ok(coefs[i]): break
        ri = mr[i]
        q = prever_demanda(precos[i:i+1], np.array([ri]), qtds[i:i+1], coefs[i:i+1], est[i:i+1], mm, pm, mdp)[0]
        c = ri*q
        if 0 < c <= rest: reb[i]=ri; rest-=c
    return reb

def sol_greedy_gmv(precos, qtds, coefs, est, mdp, orc, mm, pm):
    n = len(precos); mr = precos*mdp; reb = np.zeros(n); rest = orc
    gp = np.zeros(n)
    for i in range(n):
        if not _ok(coefs[i]): continue
        q = prever_demanda(precos[i:i+1], mr[i:i+1], qtds[i:i+1], coefs[i:i+1], est[i:i+1], mm, pm, mdp)[0]
        gp[i] = (precos[i]-mr[i])*q
    for i in np.argsort(-gp):
        if rest <= 0 or not _ok(coefs[i]): break
        q = prever_demanda(precos[i:i+1], np.array([mr[i]]), qtds[i:i+1], coefs[i:i+1], est[i:i+1], mm, pm, mdp)[0]
        c = mr[i]*q
        if 0 < c <= rest: reb[i]=mr[i]; rest-=c
    return reb

def sol_rand(precos, coefs, mdp):
    mr = precos*mdp; mask = coefs<0; r = np.zeros(len(precos))
    r[mask] = np.random.uniform(0,1,mask.sum())*mr[mask]
    return r

def projetar(reb, precos, qtds, coefs, est, orc, mdp, mm, pm):
    reb = np.clip(reb, 0, precos*mdp); reb[coefs>=0]=0.0
    _, _, _, custo, _ = calc_metricas(precos, reb, qtds, coefs, est, mm, pm, mdp)
    if custo > orc and custo > 0:
        reb *= orc/custo*0.95; reb = np.clip(reb, 0, precos*mdp); reb[coefs>=0]=0.0
    return reb

# =====================================================================
# SIMULATED ANNEALING
# =====================================================================
def gerar_vizinho(reb, precos, qtds, coefs, est, pc, ips, mdp, orc, ip_alvo, temp, t0, mm, pm):
    novo = reb.copy(); n = len(reb); fator = temp/t0; mr = precos*mdp
    np_ = max(1, int(min(n, max(3, n*0.1))*fator+1))
    for i in random.sample(range(n), min(np_, n)):
        if mr[i]<=0 or coefs[i]>=0: novo[i]=0.0; continue
        r = random.random()
        if r<0.05: novo[i]=random.uniform(0,mr[i])
        elif r<0.10: novo[i]=0.0
        elif r<0.15: novo[i]=mr[i]
        elif r<0.25:
            rip = min(max(0.0, precos[i]-ip_alvo*pc[i]), mr[i])
            novo[i]=np.clip(rip+random.gauss(0,max(mr[i]*fator*0.15,0.001)),0,mr[i])
        elif r<0.35:
            j = random.randint(0,n-1)
            if j!=i and coefs[j]<0:
                t=novo[i]+novo[j]; s=random.uniform(0,1)
                novo[i]=min(s*t,mr[i]); novo[j]=min((1-s)*t,mr[j])
        else:
            sig = mr[i]*fator*0.3+mr[i]*0.02
            novo[i]=np.clip(novo[i]+random.gauss(0,max(sig,0.001)),0,mr[i])
    novo = np.maximum(novo, 0.0)
    return projetar(novo, precos, qtds, coefs, est, orc, mdp, mm, pm)

def simulated_annealing(df, config):
    if config.seed is not None: random.seed(config.seed); np.random.seed(config.seed)
    precos=df["preco_por_atual"].values; qtds=df["quantidadeitens"].values
    coefs=df["coef"].values; est=df["estoque"].values
    ips=df["IP"].values; pc=df["preco_concorrente"].values
    taxa=config.taxa_resfriamento; mm=config.max_mult_demanda; pm=config.piso_demanda_min; mdp=config.max_desconto_pct; n=len(df)

    sols = [
        ("Greedy IP", sol_greedy_ip(precos,qtds,coefs,est,ips,pc,mdp,config.orcamento_rebate,config.ip_alvo,mm,pm)),
        ("Greedy Elasticidade", sol_greedy_elast(precos,qtds,coefs,est,mdp,config.orcamento_rebate,mm,pm)),
        ("Greedy GMV", sol_greedy_gmv(precos,qtds,coefs,est,mdp,config.orcamento_rebate,mm,pm)),
        ("Aleatoria 1", sol_rand(precos,coefs,mdp)),
        ("Aleatoria 2", sol_rand(precos,coefs,mdp)),
    ]
    for i,(nm,s) in enumerate(sols):
        sols[i] = (nm, projetar(np.maximum(s,0), precos, qtds, coefs, est, config.orcamento_rebate, mdp, mm, pm))

    print("\n"+"="*80)
    print("  SIMULATED ANNEALING v7 - Cap Proporcional ao Desconto")
    print("="*80)
    print(f"  Produtos: {n:,} ({(coefs<0).sum():,} elasticos, {(coefs>=0).sum():,} sem efeito)")
    print(f"  Orcamento: R$ {config.orcamento_rebate:,.2f} | Max desc: {mdp*100:.1f}%")
    print(f"  Cap: {mm:.0f}x PROPORCIONAL | Piso: {pm:.0f} un PROPORCIONAL")
    print(f"  IP alvo: {config.ip_alvo:.2f} | Restarts: {config.n_restarts} | Iter/restart: {config.iter_por_restart:,}")
    print("="*80)

    print(f"\n  Solucoes iniciais:")
    for nm,s in sols:
        sc=func_obj(precos,s,qtds,coefs,est,pc,ips,config)
        _,_,g,c,_=calc_metricas(precos,s,qtds,coefs,est,mm,pm,mdp)
        print(f"    {nm:<25} Score:{sc:>14,.2f} | GMV:R${g:>14,.2f} | Rebate:R${c:>10,.2f}")

    best_reb=None; best_sc=-float("inf"); hist=[]; it_total=0

    for ri in range(config.n_restarts):
        nm,si = sols[ri % len(sols)]
        cur=si.copy(); cur_sc=func_obj(precos,cur,qtds,coefs,est,pc,ips,config)
        br=cur.copy(); br_sc=cur_sc; temp=config.temp_inicial; stag=0
        print(f"\n  --- Restart {ri+1}/{config.n_restarts} [{nm}] ---")
        li = max(config.iter_por_restart//5, 1)

        for it in range(config.iter_por_restart):
            nv = gerar_vizinho(cur,precos,qtds,coefs,est,pc,ips,mdp,config.orcamento_rebate,config.ip_alvo,temp,config.temp_inicial,mm,pm)
            nv_sc = func_obj(precos,nv,qtds,coefs,est,pc,ips,config)
            d = nv_sc - cur_sc
            if d>0 or random.random()<math.exp(d/max(temp,1e-10)):
                cur=nv; cur_sc=nv_sc
                if cur_sc>br_sc: br_sc=cur_sc; br=cur.copy(); stag=0
                else: stag+=1
            else: stag+=1
            temp*=taxa; it_total+=1

            if it%li==0:
                _,_,g,c,r=calc_metricas(precos,br,qtds,coefs,est,mm,pm,mdp)
                pct=it/config.iter_por_restart*100
                hist.append({"restart":ri,"iter":it_total,"temp":temp,"score":br_sc,"gmv":g,"rebate":c})
                print(f"    [{pct:5.1f}%] Score{br_sc:>14,.2f} | GMV R${g:>14,.2f} | Rebate R${c:>10,.2f} | Temp{temp:>10.2f}")

            if stag > config.iter_por_restart//10:
                sr=sol_rand(precos,coefs,mdp)
                sr=projetar(np.maximum(sr,0),precos,qtds,coefs,est,config.orcamento_rebate,mdp,mm,pm)
                cur=0.7*br+0.3*sr; cur=np.maximum(cur,0)
                cur=projetar(cur,precos,qtds,coefs,est,config.orcamento_rebate,mdp,mm,pm)
                cur_sc=func_obj(precos,cur,qtds,coefs,est,pc,ips,config); stag=0

        if br_sc>best_sc: best_sc=br_sc; best_reb=br.copy()
        _,_,g,c,_=calc_metricas(precos,br,qtds,coefs,est,mm,pm,mdp)
        print(f"  Restart {ri+1} fim: Score{br_sc:>14,.2f} | GMV R${g:>14,.2f} | Rebate R${c:>10,.2f}")

    print("\n"+"="*80)
    _,_,gf,cf,rf=calc_metricas(precos,best_reb,qtds,coefs,est,mm,pm,mdp)
    print(f"  MELHOR: Score{best_sc:>14,.2f} | GMV:R${gf:>14,.2f} | Rebate:R${cf:>10,.2f} | Receita:R${rf:>14,.2f}")
    print("="*80)
    return best_reb, hist

# =====================================================================
# RELATORIO
# =====================================================================
def gerar_relatorio(df, df_ine, rebates, config):
    precos=df["preco_por_atual"].values; qtds=df["quantidadeitens"].values
    coefs=df["coef"].values; est=df["estoque"].values
    ips=df["IP"].values; pc=df["preco_concorrente"].values; reb_atual=df["rebate_prazo_magalu"].values
    rebates=np.maximum(rebates,0.0); mm=config.max_mult_demanda; pm=config.piso_demanda_min; mdp=config.max_desconto_pct

    po,qc,gmv_t,custo_t,receita_t = calc_metricas(precos,rebates,qtds,coefs,est,mm,pm,mdp)
    _,qs,gmv_s,_,_ = calc_metricas(precos,np.zeros(len(df)),qtds,coefs,est,mm,pm,mdp)

    ip_novo = po/np.maximum(pc,0.01)
    custo_pp = np.maximum(rebates*qc, 0.0)
    gmv_pp_com = po*qc; gmv_pp_sem = precos*qs

    resultado = pd.DataFrame({
        "navigation_id": df["navigation_id"].values,
        "categoria_comercial": df["categoria_comercial"].values,
        "seller_id": df["SellerID"].values,
        "concorrente": df["Concorrente"].values if "Concorrente" in df.columns else "N/A",
        "carteira": df["Carteira"].values if "Carteira" in df.columns else "N/A",
        "preco_atual": precos.round(2),
        "preco_concorrente": pc.round(2),
        "IP_atual": ips.round(4),
        "rebate_atual_magalu": reb_atual.round(2),
        "rebate_otimizado": rebates.round(2),
        "desconto_pct": ((rebates/np.maximum(precos,0.01))*100).round(2),
        "preco_otimizado": po.round(2),
        "IP_novo": ip_novo.round(4),
        "delta_IP": (ip_novo - ips).round(4),
        "status_ip": np.where(ip_novo<=config.ip_alvo,"COMPETITIVO",np.where(ip_novo<=config.ip_alvo*1.05,"MARGINAL","NAO_COMPETITIVO")),
        "qtd_atual": qtds.round(0),
        "qtd_prevista": qc.round(1),
        "delta_qtd": (qc-qtds).round(1),
        "mult_demanda": np.where(qtds>0, (qc/qtds).round(2), 0.0),
        "gmv_atual": gmv_pp_sem.round(2),
        "gmv_otimizado": gmv_pp_com.round(2),
        "delta_gmv": (gmv_pp_com-gmv_pp_sem).round(2),
        "custo_rebate": custo_pp.round(2),
        "lucro_previsto": (gmv_pp_com-custo_pp).round(2),
        "roi_rebate": np.where(custo_pp>0.01, np.round((gmv_pp_com-gmv_pp_sem)/np.maximum(custo_pp,0.01),2), 0.0),
        "estoque": est,
    })
    resultado = resultado.sort_values("gmv_otimizado", ascending=False).reset_index(drop=True)

    nr = (rebates>0.01).sum()
    mk = resultado["rebate_otimizado"]>0.01
    dm = resultado.loc[mk,"desconto_pct"].mean() if mk.any() else 0.0
    ipa = (ips<=config.ip_alvo).sum(); ipd = (ip_novo<=config.ip_alvo).sum()
    rat = reb_atual.sum()

    print("\n"+"="*80)
    print("  SUMARIO FINAL DA OTIMIZACAO")
    print("="*80)
    print(f"  {'Produtos elegiveis:':<40} {len(df):>10,}")
    print(f"  {'Produtos inelegiveis:':<40} {len(df_ine):>10,}")
    print(f"  {'Produtos com rebate > 0:':<40} {nr:>10,}")
    print(f"  {'Produtos com coef < 0 (elasticos):':<40} {(coefs<0).sum():>10,}")
    print("-"*80)
    print(f"  {'GMV ATUAL:':<40} R$ {gmv_s:>14,.2f}")
    print(f"  {'GMV OTIMIZADO:':<40} R$ {gmv_t:>14,.2f}")
    print(f"  {'DELTA GMV (+):':<40} R$ {gmv_t-gmv_s:>14,.2f}")
    if gmv_s>0: print(f"  {'Crescimento GMV:':<40} {(gmv_t-gmv_s)/gmv_s*100:>13.2f}%")
    print("-"*80)
    print(f"  {'Rebate ATUAL (rebate_prazo_magalu):':<40} R$ {rat:>14,.2f}")
    print(f"  {'Rebate OTIMIZADO (novo):':<40} R$ {custo_t:>14,.2f}")
    print(f"  {'Orcamento disponivel:':<40} R$ {config.orcamento_rebate:>14,.2f}")
    print(f"  {'Orcamento utilizado:':<40} {custo_t/config.orcamento_rebate*100:>13.2f}%")
    print("-"*80)
    print(f"  {'Receita Liquida (GMV-Rebate):':<40} R$ {receita_t:>14,.2f}")
    print(f"  {'Desconto Medio (c/ rebate):':<40} {dm:>13.2f}%")
    if custo_t>0: print(f"  {'ROI do Rebate (deltaGMV/custo):':<40} {(gmv_t-gmv_s)/custo_t:>13.2f}x")
    print("-"*80)
    print(f"  {'IP:':<40}")
    print(f"  {'  Competitivos ANTES:':<40} {ipa:>10,}")
    print(f"  {'  Competitivos DEPOIS:':<40} {ipd:>10,}")
    print(f"  {'  IPs corrigidos:':<40} {ipd-ipa:>10,}")
    print(f"  {'  IP medio antes:':<40} {ips.mean():>13.4f}")
    print(f"  {'  IP medio depois:':<40} {ip_novo.mean():>13.4f}")
    print("="*80)
    return resultado

def otimizar_rebate(df_input, orcamento_rebate, max_desconto_pct=0.10, max_mult_demanda=2.0, piso_demanda_min=3.0, ip_alvo=1.0, max_iter=1_000_000, n_restarts=5, seed=42, is_spark=True):
    config = OptConfig(orcamento_rebate=orcamento_rebate, max_desconto_pct=max_desconto_pct, max_mult_demanda=max_mult_demanda, piso_demanda_min=piso_demanda_min, ip_alvo=ip_alvo, max_iter=max_iter, n_restarts=n_restarts, seed=seed)
    df = preparar_dados(df_input, is_spark=is_spark)
    df_el, df_ine = filtrar_elegiveis(df)
    diagnostico(df_el, df_ine)
    if len(df_el)==0: print("\n  ERRO: Nenhum produto elegivel!"); return pd.DataFrame(), []
    rebates, hist = simulated_annealing(df_el, config)
    resultado = gerar_relatorio(df_el, df_ine, rebates, config)
    return resultado, hist

# =====================================================================
# EXECUCAO COM df_final
# =====================================================================
resultado_ud_diario, hist_ud_d = otimizar_rebate(
    df_input=df_ud_diario,
    orcamento_rebate=5_000,
    max_desconto_pct=0.10,
    max_mult_demanda=2.0,
    piso_demanda_min=3.0,
    ip_alvo=0.98,
    max_iter=1_000_000,
    n_restarts=5,
    seed=42,
    is_spark=True,
)

print("\n  TOP 30 PRODUTOS POR GMV OTIMIZADO:")
print("-"*190)
cols = ["navigation_id","categoria_comercial","preco_atual","preco_concorrente",
        "IP_atual","rebate_otimizado","desconto_pct","preco_otimizado","IP_novo",
        "status_ip","qtd_atual","qtd_prevista","mult_demanda","gmv_otimizado",
        "custo_rebate","roi_rebate"]
print(resultado_ud_diario[cols].head(30).to_string(index=False, float_format="%.2f"))

print("\n\n  SUMARIO POR CATEGORIA:")
print("-"*140)
sumario = resultado_ud_diario.groupby("categoria_comercial").agg(
    n_produtos=("navigation_id","count"),
    rebate_total=("custo_rebate","sum"),
    gmv_atual=("gmv_atual","sum"),
    gmv_otimizado=("gmv_otimizado","sum"),
    delta_gmv=("delta_gmv","sum"),
    desc_medio=("desconto_pct","mean"),
    mult_medio=("mult_demanda","mean"),
    ip_medio_novo=("IP_novo","mean"),
    n_competitivos=("status_ip", lambda x: (x=="COMPETITIVO").sum()),
).round(2)
print(sumario.to_string())

print("\n\n  ANALISE DE IP POR STATUS:")
print("-"*70)
ip_a = resultado_ud_diario.groupby("status_ip").agg(
    n_produtos=("navigation_id","count"),
    ip_medio=("IP_novo","mean"),
    rebate_medio=("rebate_otimizado","mean"),
    gmv_total=("gmv_otimizado","sum"),
).round(2)
print(ip_a.to_string())

# Salvar
resultado_ud_diario.to_csv("resultado_ud_diario.csv", index=False)
print("\nSalvo: resultado_ud_diario.csv")

                                                                                ]]]00]


  DIAGNOSTICO DOS DADOS
  Total: 9,202 | Elegiveis: 9,202 | Inelegiveis: 0

  Precos: Min R$2.98 | Med R$75.90 | Max R$15,661.19
  Vendas: Min 1 | Med 1 | Max 981 | Total 16,191
  Coef: Negativo (elastico) 8,915 | Zero (sem efeito) 287
  IP: Medio 1.0112 | >1 (nao compet.) 909 | <=1 (compet.) 8,293

  GMV atual: R$ 2,084,888.15
  Rebate atual (rebate_prazo_magalu): R$ 36,350.55


[Stage 52:=====================================================(205 + -5) / 200]


  SIMULATED ANNEALING v7 - Cap Proporcional ao Desconto
  Produtos: 9,202 (8,915 elasticos, 287 sem efeito)
  Orcamento: R$ 5,000.00 | Max desc: 10.0%
  Cap: 2x PROPORCIONAL | Piso: 3 un PROPORCIONAL
  IP alvo: 0.98 | Restarts: 5 | Iter/restart: 200,000

  Solucoes iniciais:
    Greedy IP                 Score:  2,080,390.54 | GMV:R$  2,085,390.54 | Rebate:R$  5,000.00
    Greedy Elasticidade       Score:  2,089,585.45 | GMV:R$  2,077,592.98 | Rebate:R$  4,999.84
    Greedy GMV                Score:  2,075,859.01 | GMV:R$  2,079,885.11 | Rebate:R$  4,999.99
    Aleatoria 1               Score:  2,094,126.63 | GMV:R$  2,095,456.99 | Rebate:R$  2,318.13
    Aleatoria 2               Score:  2,094,059.61 | GMV:R$  2,095,881.80 | Rebate:R$  2,309.96

  --- Restart 1/5 [Greedy IP] ---
    [  0.0%] Score  2,137,091.70 | GMV R$  2,092,859.57 | Rebate R$  3,256.17 | Temp  49995.57


[Stage 52:=====================================================(205 + -5) / 200]

...

[Stage 52:=====================================================(205 + -5) / 200]

    [ 60.0%] Score  2,814,050.02 | GMV R$  2,125,061.80 | Rebate R$  4,999.99 | Temp      1.20
    [ 80.0%] Score  2,843,393.28 | GMV R$  2,128,405.30 | Rebate R$  4,999.96 | Temp      0.03


[Stage 52:=====================================================(205 + -5) / 200]

  Restart 5 fim: Score  2,862,372.47 | GMV R$  2,130,383.58 | Rebate R$  4,999.99

  MELHOR: Score  2,871,611.58 | GMV:R$  2,129,620.38 | Rebate:R$  4,999.99 | Receita:R$  2,124,620.38

  SUMARIO FINAL DA OTIMIZACAO
  Produtos elegiveis:                           9,202
  Produtos inelegiveis:                             0
  Produtos com rebate > 0:                      6,993
  Produtos com coef < 0 (elasticos):            8,915
--------------------------------------------------------------------------------
  GMV ATUAL:                               R$   2,059,453.02
  GMV OTIMIZADO:                           R$   2,129,620.38
  DELTA GMV (+):                           R$      70,167.35
  Crescimento GMV:                                  3.41%
--------------------------------------------------------------------------------
  Rebate ATUAL (rebate_prazo_magalu):      R$      36,350.55
  Rebate OTIMIZADO (novo):                 R$       4,999.99
  Orcamento disponivel:                    

[Stage 52:=====================================================(205 + -5) / 200]

In [91]:
@dataclass
class OptConfig:
    orcamento_rebate: float
    max_desconto_pct: float
    max_mult_demanda: float = 2.0
    piso_demanda_min: float = 3.0
    ip_alvo: float = 1.0
    peso_ip: float = 5.0
    temp_inicial: float = 50000.0
    temp_final: float = 0.001
    max_iter: int = 1_000_000
    n_restarts: int = 5
    iter_por_restart: int = 0
    penalidade_orcamento: float = 100.0
    seed: Optional[int] = 42
    def __post_init__(self):
        if self.iter_por_restart == 0:
            self.iter_por_restart = self.max_iter // self.n_restarts
    @property
    def taxa_resfriamento(self) -> float:
        return (self.temp_final / self.temp_inicial) ** (1.0 / max(self.iter_por_restart, 1))

def preparar_dados(df_input, is_spark=True):
    # Mapeamento: nome atual → nome que o otimizador usa internamente
    rename_map = {
        "categoria": "categoria_comercial",
        "preco_por": "preco_por_atual",
        "concorrente": "Concorrente",
        "carteira": "Carteira",
        "rebate_unitario_medio": "rebate_prazo_magalu",
        "seller_id": "SellerID",
    }
    
    cols_obr = ["navigation_id", "categoria_comercial", "preco_por_atual",
                "quantidadeitens", "coef", "intercept", "SellerID", "estoque", "IP"]
    cols_opt = ["Concorrente", "Carteira", "rebate_prazo_magalu"]

    if is_spark:
        # Renomear colunas antes de converter
        for old, new in rename_map.items():
            if old in df_input.columns:
                df_input = df_input.withColumnRenamed(old, new)
        cols_usar = [c for c in cols_obr + cols_opt if c in df_input.columns]
        df = df_input.select(cols_usar).toPandas()
    else:
        df = df_input.rename(columns=rename_map)
        cols_usar = [c for c in cols_obr + cols_opt if c in df.columns]
        df = df[cols_usar].copy()

    df = df.dropna(subset=["coef", "preco_por_atual"])
    df = df[df["preco_por_atual"] > 0].copy()
    df["preco_por_atual"] = df["preco_por_atual"].astype(float)
    df["coef"] = df["coef"].astype(float)
    df["intercept"] = df["intercept"].fillna(0).astype(float)
    df["quantidadeitens"] = df["quantidadeitens"].fillna(0).astype(float)
    df["estoque"] = df["estoque"].fillna(9999).astype(int)
    df["IP"] = df["IP"].fillna(1.0).astype(float).replace(0.0, 1.0)
    df["preco_concorrente"] = np.where(
        df["IP"] > 0, df["preco_por_atual"] / df["IP"], df["preco_por_atual"]
    )
    for col, default in [("Concorrente", "N/A"), ("Carteira", "N/A"), ("rebate_prazo_magalu", 0.0)]:
        if col not in df.columns:
            df[col] = default
        else:
            df[col] = df[col].fillna(default)
    df["rebate_prazo_magalu"] = pd.to_numeric(df["rebate_prazo_magalu"], errors="coerce").fillna(0.0)
    df = df.drop_duplicates(subset=["navigation_id"], keep="first").reset_index(drop=True)
    return df

def filtrar_elegiveis(df):
    mask = (df["quantidadeitens"]>0) & (df["estoque"]>0) & (df["preco_por_atual"]>0)
    return df[mask].reset_index(drop=True), df[~mask].reset_index(drop=True)

def diagnostico(df, df_ine):
    print("\n" + "="*80)
    print("  DIAGNOSTICO DOS DADOS")
    print("="*80)
    print(f"  Total: {len(df)+len(df_ine):,} | Elegiveis: {len(df):,} | Inelegiveis: {len(df_ine):,}")
    if len(df_ine)>0:
        print(f"    Sem vendas: {(df_ine['quantidadeitens']<=0).sum():,} | Sem estoque: {(df_ine['estoque']<=0).sum():,}")
    print(f"\n  Precos: Min R${df['preco_por_atual'].min():,.2f} | Med R${df['preco_por_atual'].median():,.2f} | Max R${df['preco_por_atual'].max():,.2f}")
    print(f"  Vendas: Min {df['quantidadeitens'].min():.0f} | Med {df['quantidadeitens'].median():.0f} | Max {df['quantidadeitens'].max():.0f} | Total {df['quantidadeitens'].sum():,.0f}")
    c = df["coef"]
    print(f"  Coef: Negativo (elastico) {(c<0).sum():,} | Zero (sem efeito) {(c==0).sum():,}")
    print(f"  IP: Medio {df['IP'].mean():.4f} | >1 (nao compet.) {(df['IP']>1).sum():,} | <=1 (compet.) {(df['IP']<=1).sum():,}")
    print(f"\n  GMV atual: R$ {(df['preco_por_atual']*df['quantidadeitens']).sum():,.2f}")
    print(f"  Rebate atual (rebate_prazo_magalu): R$ {df['rebate_prazo_magalu'].sum():,.2f}")
    print("="*80)

# =====================================================================
# MODELO DE DEMANDA - CAP PROPORCIONAL
# =====================================================================
def prever_demanda(precos, rebates, qtds, coefs, estoques, max_mult=2.0, piso_min=3.0, max_desc_pct=0.10):
    """
    Q_nova = qtd_atual - coef * rebate  (coef negativo -> desconto aumenta demanda)

    Cap PROPORCIONAL ao desconto:
      fator = desconto_dado / desconto_maximo  (0 a 1)
      mult_max = 1 + fator * (max_mult - 1)
      piso = qtd + piso_min * fator

    Desc 10% -> mult 2.0x | Desc 5% -> mult 1.5x | Desc 1% -> mult 1.1x | Desc 0.2% -> mult 1.02x
    """
    r = np.maximum(rebates, 0.0)
    qtd_nova = qtds + coefs * (-r)
    desc_pct = r / np.maximum(precos, 0.01)
    fator = np.clip(desc_pct / max(max_desc_pct, 0.001), 0.0, 1.0)
    mult = 1.0 + fator * (max_mult - 1.0)
    piso = qtds + piso_min * fator
    max_qtd = np.maximum(qtds * mult, piso)
    teto = np.minimum(max_qtd, estoques.astype(float))
    return np.clip(qtd_nova, 0.0, teto)

def calc_metricas(precos, rebates, qtds, coefs, estoques, mm=2.0, pm=3.0, mdp=0.10):
    r = np.maximum(rebates, 0.0)
    po = np.maximum(precos - r, 0.01)
    qn = prever_demanda(precos, r, qtds, coefs, estoques, mm, pm, mdp)
    gmv = (po * qn).sum()
    custo = (r * qn).sum()
    return po, qn, gmv, max(custo, 0.0), gmv - max(custo, 0.0)

def func_obj(precos, rebates, qtds, coefs, estoques, pc, ips, cfg):
    r = np.maximum(rebates, 0.0)
    _, _, gmv, custo, receita = calc_metricas(precos, r, qtds, coefs, estoques, cfg.max_mult_demanda, cfg.piso_demanda_min, cfg.max_desconto_pct)
    score = receita
    if custo > cfg.orcamento_rebate:
        exc = custo - cfg.orcamento_rebate
        score -= cfg.penalidade_orcamento * exc + 0.1 * exc**2
    viol = np.maximum(r/np.maximum(precos,0.01) - cfg.max_desconto_pct, 0.0)
    if viol.sum() > 0: score -= 5000.0 * viol.sum()
    po = np.maximum(precos - r, 0.01)
    ip_novo = po / np.maximum(pc, 0.01)
    score += cfg.peso_ip * ((ips > cfg.ip_alvo) & (ip_novo <= cfg.ip_alvo)).sum() * 100
    score -= 0.3 * r[ips <= cfg.ip_alvo * 0.85].sum()
    return score

# =====================================================================
# SOLUCOES INICIAIS
# =====================================================================
def _ok(coef): return coef < 0

def sol_greedy_ip(precos, qtds, coefs, est, ips, pc, mdp, orc, ip_alvo, mm, pm):
    n = len(precos); mr = precos*mdp; reb = np.zeros(n); rest = orc
    for i in np.argsort(-ips):
        if rest <= 0: break
        if not _ok(coefs[i]) or ips[i] <= ip_alvo*0.9: continue
        ri = min(max(0.0, precos[i]-ip_alvo*pc[i]), mr[i])
        q = prever_demanda(precos[i:i+1], np.array([ri]), qtds[i:i+1], coefs[i:i+1], est[i:i+1], mm, pm, mdp)[0]
        c = ri*q
        if 0 < c <= rest: reb[i]=ri; rest-=c
    return reb

def sol_greedy_elast(precos, qtds, coefs, est, mdp, orc, mm, pm):
    n = len(precos); mr = precos*mdp; reb = np.zeros(n); rest = orc
    elast = np.abs(np.minimum(coefs, 0))
    for i in np.argsort(-elast):
        if rest <= 0 or not _ok(coefs[i]): break
        ri = mr[i]
        q = prever_demanda(precos[i:i+1], np.array([ri]), qtds[i:i+1], coefs[i:i+1], est[i:i+1], mm, pm, mdp)[0]
        c = ri*q
        if 0 < c <= rest: reb[i]=ri; rest-=c
    return reb

def sol_greedy_gmv(precos, qtds, coefs, est, mdp, orc, mm, pm):
    n = len(precos); mr = precos*mdp; reb = np.zeros(n); rest = orc
    gp = np.zeros(n)
    for i in range(n):
        if not _ok(coefs[i]): continue
        q = prever_demanda(precos[i:i+1], mr[i:i+1], qtds[i:i+1], coefs[i:i+1], est[i:i+1], mm, pm, mdp)[0]
        gp[i] = (precos[i]-mr[i])*q
    for i in np.argsort(-gp):
        if rest <= 0 or not _ok(coefs[i]): break
        q = prever_demanda(precos[i:i+1], np.array([mr[i]]), qtds[i:i+1], coefs[i:i+1], est[i:i+1], mm, pm, mdp)[0]
        c = mr[i]*q
        if 0 < c <= rest: reb[i]=mr[i]; rest-=c
    return reb

def sol_rand(precos, coefs, mdp):
    mr = precos*mdp; mask = coefs<0; r = np.zeros(len(precos))
    r[mask] = np.random.uniform(0,1,mask.sum())*mr[mask]
    return r

def projetar(reb, precos, qtds, coefs, est, orc, mdp, mm, pm):
    reb = np.clip(reb, 0, precos*mdp); reb[coefs>=0]=0.0
    _, _, _, custo, _ = calc_metricas(precos, reb, qtds, coefs, est, mm, pm, mdp)
    if custo > orc and custo > 0:
        reb *= orc/custo*0.95; reb = np.clip(reb, 0, precos*mdp); reb[coefs>=0]=0.0
    return reb

# =====================================================================
# SIMULATED ANNEALING
# =====================================================================
def gerar_vizinho(reb, precos, qtds, coefs, est, pc, ips, mdp, orc, ip_alvo, temp, t0, mm, pm):
    novo = reb.copy(); n = len(reb); fator = temp/t0; mr = precos*mdp
    np_ = max(1, int(min(n, max(3, n*0.1))*fator+1))
    for i in random.sample(range(n), min(np_, n)):
        if mr[i]<=0 or coefs[i]>=0: novo[i]=0.0; continue
        r = random.random()
        if r<0.05: novo[i]=random.uniform(0,mr[i])
        elif r<0.10: novo[i]=0.0
        elif r<0.15: novo[i]=mr[i]
        elif r<0.25:
            rip = min(max(0.0, precos[i]-ip_alvo*pc[i]), mr[i])
            novo[i]=np.clip(rip+random.gauss(0,max(mr[i]*fator*0.15,0.001)),0,mr[i])
        elif r<0.35:
            j = random.randint(0,n-1)
            if j!=i and coefs[j]<0:
                t=novo[i]+novo[j]; s=random.uniform(0,1)
                novo[i]=min(s*t,mr[i]); novo[j]=min((1-s)*t,mr[j])
        else:
            sig = mr[i]*fator*0.3+mr[i]*0.02
            novo[i]=np.clip(novo[i]+random.gauss(0,max(sig,0.001)),0,mr[i])
    novo = np.maximum(novo, 0.0)
    return projetar(novo, precos, qtds, coefs, est, orc, mdp, mm, pm)

def simulated_annealing(df, config):
    if config.seed is not None: random.seed(config.seed); np.random.seed(config.seed)
    precos=df["preco_por_atual"].values; qtds=df["quantidadeitens"].values
    coefs=df["coef"].values; est=df["estoque"].values
    ips=df["IP"].values; pc=df["preco_concorrente"].values
    taxa=config.taxa_resfriamento; mm=config.max_mult_demanda; pm=config.piso_demanda_min; mdp=config.max_desconto_pct; n=len(df)

    sols = [
        ("Greedy IP", sol_greedy_ip(precos,qtds,coefs,est,ips,pc,mdp,config.orcamento_rebate,config.ip_alvo,mm,pm)),
        ("Greedy Elasticidade", sol_greedy_elast(precos,qtds,coefs,est,mdp,config.orcamento_rebate,mm,pm)),
        ("Greedy GMV", sol_greedy_gmv(precos,qtds,coefs,est,mdp,config.orcamento_rebate,mm,pm)),
        ("Aleatoria 1", sol_rand(precos,coefs,mdp)),
        ("Aleatoria 2", sol_rand(precos,coefs,mdp)),
    ]
    for i,(nm,s) in enumerate(sols):
        sols[i] = (nm, projetar(np.maximum(s,0), precos, qtds, coefs, est, config.orcamento_rebate, mdp, mm, pm))

    print("\n"+"="*80)
    print("  SIMULATED ANNEALING v7 - Cap Proporcional ao Desconto")
    print("="*80)
    print(f"  Produtos: {n:,} ({(coefs<0).sum():,} elasticos, {(coefs>=0).sum():,} sem efeito)")
    print(f"  Orcamento: R$ {config.orcamento_rebate:,.2f} | Max desc: {mdp*100:.1f}%")
    print(f"  Cap: {mm:.0f}x PROPORCIONAL | Piso: {pm:.0f} un PROPORCIONAL")
    print(f"  IP alvo: {config.ip_alvo:.2f} | Restarts: {config.n_restarts} | Iter/restart: {config.iter_por_restart:,}")
    print("="*80)

    print(f"\n  Solucoes iniciais:")
    for nm,s in sols:
        sc=func_obj(precos,s,qtds,coefs,est,pc,ips,config)
        _,_,g,c,_=calc_metricas(precos,s,qtds,coefs,est,mm,pm,mdp)
        print(f"    {nm:<25} Score:{sc:>14,.2f} | GMV:R${g:>14,.2f} | Rebate:R${c:>10,.2f}")

    best_reb=None; best_sc=-float("inf"); hist=[]; it_total=0

    for ri in range(config.n_restarts):
        nm,si = sols[ri % len(sols)]
        cur=si.copy(); cur_sc=func_obj(precos,cur,qtds,coefs,est,pc,ips,config)
        br=cur.copy(); br_sc=cur_sc; temp=config.temp_inicial; stag=0
        print(f"\n  --- Restart {ri+1}/{config.n_restarts} [{nm}] ---")
        li = max(config.iter_por_restart//5, 1)

        for it in range(config.iter_por_restart):
            nv = gerar_vizinho(cur,precos,qtds,coefs,est,pc,ips,mdp,config.orcamento_rebate,config.ip_alvo,temp,config.temp_inicial,mm,pm)
            nv_sc = func_obj(precos,nv,qtds,coefs,est,pc,ips,config)
            d = nv_sc - cur_sc
            if d>0 or random.random()<math.exp(d/max(temp,1e-10)):
                cur=nv; cur_sc=nv_sc
                if cur_sc>br_sc: br_sc=cur_sc; br=cur.copy(); stag=0
                else: stag+=1
            else: stag+=1
            temp*=taxa; it_total+=1

            if it%li==0:
                _,_,g,c,r=calc_metricas(precos,br,qtds,coefs,est,mm,pm,mdp)
                pct=it/config.iter_por_restart*100
                hist.append({"restart":ri,"iter":it_total,"temp":temp,"score":br_sc,"gmv":g,"rebate":c})
                print(f"    [{pct:5.1f}%] Score{br_sc:>14,.2f} | GMV R${g:>14,.2f} | Rebate R${c:>10,.2f} | Temp{temp:>10.2f}")

            if stag > config.iter_por_restart//10:
                sr=sol_rand(precos,coefs,mdp)
                sr=projetar(np.maximum(sr,0),precos,qtds,coefs,est,config.orcamento_rebate,mdp,mm,pm)
                cur=0.7*br+0.3*sr; cur=np.maximum(cur,0)
                cur=projetar(cur,precos,qtds,coefs,est,config.orcamento_rebate,mdp,mm,pm)
                cur_sc=func_obj(precos,cur,qtds,coefs,est,pc,ips,config); stag=0

        if br_sc>best_sc: best_sc=br_sc; best_reb=br.copy()
        _,_,g,c,_=calc_metricas(precos,br,qtds,coefs,est,mm,pm,mdp)
        print(f"  Restart {ri+1} fim: Score{br_sc:>14,.2f} | GMV R${g:>14,.2f} | Rebate R${c:>10,.2f}")

    print("\n"+"="*80)
    _,_,gf,cf,rf=calc_metricas(precos,best_reb,qtds,coefs,est,mm,pm,mdp)
    print(f"  MELHOR: Score{best_sc:>14,.2f} | GMV:R${gf:>14,.2f} | Rebate:R${cf:>10,.2f} | Receita:R${rf:>14,.2f}")
    print("="*80)
    return best_reb, hist

# =====================================================================
# RELATORIO
# =====================================================================
def gerar_relatorio(df, df_ine, rebates, config):
    precos=df["preco_por_atual"].values; qtds=df["quantidadeitens"].values
    coefs=df["coef"].values; est=df["estoque"].values
    ips=df["IP"].values; pc=df["preco_concorrente"].values; reb_atual=df["rebate_prazo_magalu"].values
    rebates=np.maximum(rebates,0.0); mm=config.max_mult_demanda; pm=config.piso_demanda_min; mdp=config.max_desconto_pct

    po,qc,gmv_t,custo_t,receita_t = calc_metricas(precos,rebates,qtds,coefs,est,mm,pm,mdp)
    _,qs,gmv_s,_,_ = calc_metricas(precos,np.zeros(len(df)),qtds,coefs,est,mm,pm,mdp)

    ip_novo = po/np.maximum(pc,0.01)
    custo_pp = np.maximum(rebates*qc, 0.0)
    gmv_pp_com = po*qc; gmv_pp_sem = precos*qs

    resultado = pd.DataFrame({
        "navigation_id": df["navigation_id"].values,
        "categoria_comercial": df["categoria_comercial"].values,
        "seller_id": df["SellerID"].values,
        "concorrente": df["Concorrente"].values if "Concorrente" in df.columns else "N/A",
        "carteira": df["Carteira"].values if "Carteira" in df.columns else "N/A",
        "preco_atual": precos.round(2),
        "preco_concorrente": pc.round(2),
        "IP_atual": ips.round(4),
        "rebate_atual_magalu": reb_atual.round(2),
        "rebate_otimizado": rebates.round(2),
        "desconto_pct": ((rebates/np.maximum(precos,0.01))*100).round(2),
        "preco_otimizado": po.round(2),
        "IP_novo": ip_novo.round(4),
        "delta_IP": (ip_novo - ips).round(4),
        "status_ip": np.where(ip_novo<=config.ip_alvo,"COMPETITIVO",np.where(ip_novo<=config.ip_alvo*1.05,"MARGINAL","NAO_COMPETITIVO")),
        "qtd_atual": qtds.round(0),
        "qtd_prevista": qc.round(1),
        "delta_qtd": (qc-qtds).round(1),
        "mult_demanda": np.where(qtds>0, (qc/qtds).round(2), 0.0),
        "gmv_atual": gmv_pp_sem.round(2),
        "gmv_otimizado": gmv_pp_com.round(2),
        "delta_gmv": (gmv_pp_com-gmv_pp_sem).round(2),
        "custo_rebate": custo_pp.round(2),
        "lucro_previsto": (gmv_pp_com-custo_pp).round(2),
        "roi_rebate": np.where(custo_pp>0.01, np.round((gmv_pp_com-gmv_pp_sem)/np.maximum(custo_pp,0.01),2), 0.0),
        "estoque": est,
    })
    resultado = resultado.sort_values("gmv_otimizado", ascending=False).reset_index(drop=True)

    nr = (rebates>0.01).sum()
    mk = resultado["rebate_otimizado"]>0.01
    dm = resultado.loc[mk,"desconto_pct"].mean() if mk.any() else 0.0
    ipa = (ips<=config.ip_alvo).sum(); ipd = (ip_novo<=config.ip_alvo).sum()
    rat = reb_atual.sum()

    print("\n"+"="*80)
    print("  SUMARIO FINAL DA OTIMIZACAO")
    print("="*80)
    print(f"  {'Produtos elegiveis:':<40} {len(df):>10,}")
    print(f"  {'Produtos inelegiveis:':<40} {len(df_ine):>10,}")
    print(f"  {'Produtos com rebate > 0:':<40} {nr:>10,}")
    print(f"  {'Produtos com coef < 0 (elasticos):':<40} {(coefs<0).sum():>10,}")
    print("-"*80)
    print(f"  {'GMV ATUAL:':<40} R$ {gmv_s:>14,.2f}")
    print(f"  {'GMV OTIMIZADO:':<40} R$ {gmv_t:>14,.2f}")
    print(f"  {'DELTA GMV (+):':<40} R$ {gmv_t-gmv_s:>14,.2f}")
    if gmv_s>0: print(f"  {'Crescimento GMV:':<40} {(gmv_t-gmv_s)/gmv_s*100:>13.2f}%")
    print("-"*80)
    print(f"  {'Rebate ATUAL (rebate_prazo_magalu):':<40} R$ {rat:>14,.2f}")
    print(f"  {'Rebate OTIMIZADO (novo):':<40} R$ {custo_t:>14,.2f}")
    print(f"  {'Orcamento disponivel:':<40} R$ {config.orcamento_rebate:>14,.2f}")
    print(f"  {'Orcamento utilizado:':<40} {custo_t/config.orcamento_rebate*100:>13.2f}%")
    print("-"*80)
    print(f"  {'Receita Liquida (GMV-Rebate):':<40} R$ {receita_t:>14,.2f}")
    print(f"  {'Desconto Medio (c/ rebate):':<40} {dm:>13.2f}%")
    if custo_t>0: print(f"  {'ROI do Rebate (deltaGMV/custo):':<40} {(gmv_t-gmv_s)/custo_t:>13.2f}x")
    print("-"*80)
    print(f"  {'IP:':<40}")
    print(f"  {'  Competitivos ANTES:':<40} {ipa:>10,}")
    print(f"  {'  Competitivos DEPOIS:':<40} {ipd:>10,}")
    print(f"  {'  IPs corrigidos:':<40} {ipd-ipa:>10,}")
    print(f"  {'  IP medio antes:':<40} {ips.mean():>13.4f}")
    print(f"  {'  IP medio depois:':<40} {ip_novo.mean():>13.4f}")
    print("="*80)
    return resultado

def otimizar_rebate(df_input, orcamento_rebate, max_desconto_pct=0.10, max_mult_demanda=2.0, piso_demanda_min=3.0, ip_alvo=1.0, max_iter=1_000_000, n_restarts=5, seed=42, is_spark=True):
    config = OptConfig(orcamento_rebate=orcamento_rebate, max_desconto_pct=max_desconto_pct, max_mult_demanda=max_mult_demanda, piso_demanda_min=piso_demanda_min, ip_alvo=ip_alvo, max_iter=max_iter, n_restarts=n_restarts, seed=seed)
    df = preparar_dados(df_input, is_spark=is_spark)
    df_el, df_ine = filtrar_elegiveis(df)
    diagnostico(df_el, df_ine)
    if len(df_el)==0: print("\n  ERRO: Nenhum produto elegivel!"); return pd.DataFrame(), []
    rebates, hist = simulated_annealing(df_el, config)
    resultado = gerar_relatorio(df_el, df_ine, rebates, config)
    return resultado, hist

# =====================================================================
# EXECUCAO COM df_final
# =====================================================================
resultado_ep_diario, hist_ep_d = otimizar_rebate(
    df_input=df_ep_diario,
    orcamento_rebate=5_000,
    max_desconto_pct=0.10,
    max_mult_demanda=2.0,
    piso_demanda_min=3.0,
    ip_alvo=0.98,
    max_iter=1_000_000,
    n_restarts=5,
    seed=42,
    is_spark=True,
)

print("\n  TOP 30 PRODUTOS POR GMV OTIMIZADO:")
print("-"*190)
cols = ["navigation_id","categoria_comercial","preco_atual","preco_concorrente",
        "IP_atual","rebate_otimizado","desconto_pct","preco_otimizado","IP_novo",
        "status_ip","qtd_atual","qtd_prevista","mult_demanda","gmv_otimizado",
        "custo_rebate","roi_rebate"]
print(resultado_ep_diario[cols].head(30).to_string(index=False, float_format="%.2f"))

print("\n\n  SUMARIO POR CATEGORIA:")
print("-"*140)
sumario = resultado_ep_diario.groupby("categoria_comercial").agg(
    n_produtos=("navigation_id","count"),
    rebate_total=("custo_rebate","sum"),
    gmv_atual=("gmv_atual","sum"),
    gmv_otimizado=("gmv_otimizado","sum"),
    delta_gmv=("delta_gmv","sum"),
    desc_medio=("desconto_pct","mean"),
    mult_medio=("mult_demanda","mean"),
    ip_medio_novo=("IP_novo","mean"),
    n_competitivos=("status_ip", lambda x: (x=="COMPETITIVO").sum()),
).round(2)
print(sumario.to_string())

print("\n\n  ANALISE DE IP POR STATUS:")
print("-"*70)
ip_a = resultado_ep_diario.groupby("status_ip").agg(
    n_produtos=("navigation_id","count"),
    ip_medio=("IP_novo","mean"),
    rebate_medio=("rebate_otimizado","mean"),
    gmv_total=("gmv_otimizado","sum"),
).round(2)
print(ip_a.to_string())

# Salvar
resultado_ep_diario.to_csv("resultado_ep_diario.csv", index=False)
print("\nSalvo: resultado_ep_diario.csv")

                                                                                ]]]]


  DIAGNOSTICO DOS DADOS
  Total: 4,811 | Elegiveis: 4,811 | Inelegiveis: 0

  Precos: Min R$3.12 | Med R$209.90 | Max R$12,115.50
  Vendas: Min 1 | Med 1 | Max 131 | Total 8,097
  Coef: Negativo (elastico) 4,212 | Zero (sem efeito) 599
  IP: Medio 1.0386 | >1 (nao compet.) 1,295 | <=1 (compet.) 3,516

  GMV atual: R$ 3,107,041.40
  Rebate atual (rebate_prazo_magalu): R$ 49,222.00


[Stage 52:=====================================================(205 + -5) / 200]


  SIMULATED ANNEALING v7 - Cap Proporcional ao Desconto
  Produtos: 4,811 (4,212 elasticos, 599 sem efeito)
  Orcamento: R$ 5,000.00 | Max desc: 10.0%
  Cap: 2x PROPORCIONAL | Piso: 3 un PROPORCIONAL
  IP alvo: 0.98 | Restarts: 5 | Iter/restart: 200,000

  Solucoes iniciais:
    Greedy IP                 Score:  3,116,649.10 | GMV:R$  3,119,648.97 | Rebate:R$  4,999.88
    Greedy Elasticidade       Score:  3,115,303.50 | GMV:R$  3,110,812.53 | Rebate:R$  4,999.59
    Greedy GMV                Score:  3,117,979.16 | GMV:R$  3,122,478.91 | Rebate:R$  4,999.75
    Aleatoria 1               Score:  3,125,755.80 | GMV:R$  3,127,940.48 | Rebate:R$  2,170.37
    Aleatoria 2               Score:  3,126,601.59 | GMV:R$  3,128,316.82 | Rebate:R$  2,201.10

  --- Restart 1/5 [Greedy IP] ---
    [  0.0%] Score  3,130,252.48 | GMV R$  3,124,011.18 | Rebate R$  2,748.92 | Temp  49995.57


[Stage 52:=====================================================(205 + -5) / 200]

...

[Stage 52:=====================================================(205 + -5) / 200]

    [ 40.0%] Score  3,409,000.63 | GMV R$  3,171,997.66 | Rebate R$  4,982.46 | Temp     41.62
    [ 60.0%] Score  3,447,240.49 | GMV R$  3,177,253.57 | Rebate R$  4,999.97 | Temp      1.20


[Stage 52:=====================================================(205 + -5) / 200]

    [ 80.0%] Score  3,456,181.11 | GMV R$  3,180,693.94 | Rebate R$  4,999.63 | Temp      0.03
  Restart 5 fim: Score  3,463,905.72 | GMV R$  3,182,918.58 | Rebate R$  4,999.99

  MELHOR: Score  3,475,944.93 | GMV:R$  3,182,957.74 | Rebate:R$  4,999.98 | Receita:R$  3,177,957.75

  SUMARIO FINAL DA OTIMIZACAO
  Produtos elegiveis:                           4,811
  Produtos inelegiveis:                             0
  Produtos com rebate > 0:                      3,636
  Produtos com coef < 0 (elasticos):            4,212
--------------------------------------------------------------------------------
  GMV ATUAL:                               R$   3,090,095.55
  GMV OTIMIZADO:                           R$   3,182,957.74
  DELTA GMV (+):                           R$      92,862.18
  Crescimento GMV:                                  3.01%
--------------------------------------------------------------------------------
  Rebate ATUAL (rebate_prazo_magalu):      R$      49,222.00
  Rebate 

[Stage 52:=====================================================(205 + -5) / 200]